# 🔗 Analiz 1: Cascade (Zincirleme) Arıza Analizi — Tam Kanıt Zinciri

**Hedef:** Bir arızadan sonra başka arıza tetikleniyor mu? Hangi pattern ML modeline güçlü feature olur?

**Yöntem:** Şartlama yok, veri konuşur. Her iddiayı istatistiksel testle destekle.

---

## 1. Veri Yükleme + Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Veri yukleme
df = pd.read_csv('../panel_data/temiz_veri/ariza_model.csv')
df['OLAYTARIHI'] = pd.to_datetime(df['OLAYTARIHI'], format='mixed')
df = df.sort_values(['KAPINO', 'OLAYTARIHI']).reset_index(drop=True)

print('=== TEMEL ISTATISTIK ===')
print(f'Toplam ariza: {len(df):,}')
print(f'Benzersiz arac: {df["KAPINO"].nunique():,}')
print(f'Tarih araligi: {df["OLAYTARIHI"].min()} -> {df["OLAYTARIHI"].max()}')
print(f'Veri penceresi: {(df["OLAYTARIHI"].max() - df["OLAYTARIHI"].min()).days} gun')
print()
print(f'Ciddi ariza (binary): {df["ciddi_ariza"].mean()*100:.1f}%')
print(f'Ciddiyet skoru: mean={df["ciddiyet_skoru"].mean():.3f}, std={df["ciddiyet_skoru"].std():.3f}')
print()
print(f'Kategori sayisi: {df["ARIZAUSTKODTANIM"].nunique()}')

# CASCADE ICIN TEMEL FEATURE: ardisik ariza arasi sure
df['onceki_ariza'] = df.groupby('KAPINO')['OLAYTARIHI'].shift(1)
df['onceki_kategori'] = df.groupby('KAPINO')['ARIZAUSTKODTANIM'].shift(1)
df['saat_farki']  = (df['OLAYTARIHI'] - df['onceki_ariza']).dt.total_seconds() / 3600
df['gun_farki']   = df['saat_farki'] / 24

print()
print('=== ARDISIK ARIZA SURE FARKLARI ===')
print(df['gun_farki'].describe([0.10, 0.25, 0.50, 0.75, 0.90]).round(2).to_string())


=== TEMEL ISTATISTIK ===
Toplam ariza: 58,559
Benzersiz arac: 3,509
Tarih araligi: 2025-01-01 00:30:56 -> 2025-06-30 23:29:56
Veri penceresi: 180 gun

Ciddi ariza (binary): 38.0%
Ciddiyet skoru: mean=3.610, std=1.859

Kategori sayisi: 35

=== ARDISIK ARIZA SURE FARKLARI ===
count    55050.00
mean         9.37
std         12.36
min          0.00
10%          0.63
25%          1.85
50%          5.14
75%         11.97
90%         23.08
max        175.75


---

## 2. Cascade Eşik Analizi
Ardışık arıza süre dağılımına göre "cascade" tanımı için doğal eşik bul.

In [2]:
# BOLUM 2: Cascade Esik Analizi — Veri konussun
# Soru: "Cascade" demek icin ardisik arıza arasi sure kac saat/gun olmali?
# Yontem: Sure dagilimini gor, dogal kirilma noktasini bul.

valid = df[df['saat_farki'].notna()].copy()

# Histogram: kac saat icinde tekrar ariza var?
print('=== ARDISIK ARIZA SURE DAGILIMI ===')
print(f'Total ardisik kayit: {len(valid):,}')
print()
print(f'{"Esik":15s}  pct_altinda  cumulative_n')
for esik_saat in [1, 6, 24, 72, 168, 720]:
    n_alt = (valid['saat_farki'] < esik_saat).sum()
    pct = n_alt / len(valid) * 100
    label = f'< {esik_saat}s' if esik_saat < 24 else f'< {esik_saat//24}g'
    print(f'{label:15s}  {pct:6.2f}%      {n_alt:8d}')

# Histogram gorselle
sub = valid[valid['saat_farki'] <= 168]  # 7 gun ic histogram
fig = px.histogram(sub, x='saat_farki', nbins=50,
    title='Ardisik Ariza Arasi Sure (7 gun penceresi)',
    labels={'saat_farki': 'Saat farki'})
fig.add_vline(x=24, line_dash='dash', line_color='red', annotation_text='24s')
fig.add_vline(x=168, line_dash='dash', line_color='orange', annotation_text='7g')
fig.show()

# CASCADE TANIMI: 24 saat icinde tekrar = cascade (yaygin convention)
# Veriye gore inceleyelim: 24s esiginde anormal yogunlasma var mi?
# Beklenti: random olsa exponential dagilim ile uyumlu olmali

# Median ardisik sure ile karsilastir
ort_sure = valid['saat_farki'].median()
print()
print(f'Ardisik median sure: {ort_sure:.1f} saat ({ort_sure/24:.2f} gun)')

# Cascade tanim ESIK = 24 saat (1 gun)
CASCADE_ESIK_SAAT = 24
df['cascade'] = df['saat_farki'] < CASCADE_ESIK_SAAT

print()
print(f'=== CASCADE TANIMI (esik {CASCADE_ESIK_SAAT}s) ===')
print(f'Cascade arizalar: {df["cascade"].sum():,} ({df["cascade"].mean()*100:.1f}%)')
print(f'Stand-alone arizalar: {(~df["cascade"]).sum():,}')


=== ARDISIK ARIZA SURE DAGILIMI ===
Total ardisik kayit: 55,050

Esik             pct_altinda  cumulative_n
< 1s               0.70%           386
< 6s               5.48%          3018
< 1g              15.85%          8727
< 3g              35.57%         19584
< 7g              59.25%         32615
< 30g             93.87%         51675



Ardisik median sure: 123.3 saat (5.14 gun)

=== CASCADE TANIMI (esik 24s) ===
Cascade arizalar: 8,727 (14.9%)
Stand-alone arizalar: 49,832


---

## 3. Cascade Prevalansı (Random Null Model)
Gözlemlenen cascade oranı rastgele beklentiden yüksek mi? Poisson null model ile karşılaştır.

In [3]:
# BOLUM 3: Cascade Prevalansi ve Random Null Karsilastirmasi
# Soru: Gozlemlenen cascade orani rastgele beklentiden yuksek mi?
# Yontem: Random null model — eger arizalar rastgele dagilsa cascade orani ne olurdu?

# Gozlemlenen cascade orani
gercek_cascade = df['cascade'].sum()
gercek_orani = df['cascade'].mean()

# Random null model:
# Her arac icin: o aracin TOPLAM ariza sayisi sabit, ama TARIHLER veri penceresinde rastgele dagitilsa
# 24 saat icinde tekrar olasiligi nedir?

# Beklenti: araç başına lambda = N_ariza / N_gun (Poisson)
# 24 saat icinde tekrar P(N>=2 | N>=1) = 1 - P(N=1|N>=1) (Poisson)

veri_gun = (df['OLAYTARIHI'].max() - df['OLAYTARIHI'].min()).days
arac_ariza_n = df.groupby('KAPINO').size()
arac_lambda  = arac_ariza_n / veri_gun  # ariza/gun
arac_lambda_24s = arac_lambda  # 1 gun = 24 saat

# Beklenen cascade: her ardisik cift icin P(diğer 24s icinde) = 1 - exp(-lambda_24s)
# Simple estimate: tum araclar uzerinden ortalama
ort_lambda = arac_lambda_24s.mean()
beklenen_cascade_orani = 1 - np.exp(-ort_lambda)

print('=== RANDOM NULL MODEL ===')
print(f'Veri penceresi: {veri_gun} gun')
print(f'Ort. ariza/arac/gun (lambda): {ort_lambda:.4f}')
print(f'Random beklenen cascade orani (24s pencere): {beklenen_cascade_orani*100:.2f}%')
print()
print(f'=== GOZLEMLENEN vs BEKLENEN ===')
print(f'Gozlemlenen cascade orani: {gercek_orani*100:.2f}%')
print(f'Beklenen (random):         {beklenen_cascade_orani*100:.2f}%')
print(f'Oran (gozlemlenen/beklenen): x{gercek_orani/beklenen_cascade_orani:.2f}')

# Z-test
# H0: gercek_orani == beklenen_orani
n = (df['onceki_ariza'].notna()).sum()
se = np.sqrt(beklenen_cascade_orani * (1-beklenen_cascade_orani) / n)
z = (gercek_orani - beklenen_cascade_orani) / se
from scipy.stats import norm
p = 2 * (1 - norm.cdf(abs(z)))
print(f'Z-test: z={z:.2f}, p={p:.6f}')

if p < 0.05:
    if gercek_orani > beklenen_cascade_orani:
        print(f'VERIYE GORE: Cascade orani random beklentinin {gercek_orani/beklenen_cascade_orani:.2f}x USTUNDE (p<0.05).')
        print('  -> Arizalar rastgele degil — gercek bir cascade etkisi var.')
    else:
        print(f'VERIYE GORE: Cascade orani random beklentinin ALTINDA (p<0.05).')
        print('  -> Tekrar arizalar onlenebiliyor olabilir.')
else:
    print(f'VERIYE GORE: Cascade orani random beklentiyle FARKSIZ (p={p:.4f}).')


=== RANDOM NULL MODEL ===
Veri penceresi: 180 gun
Ort. ariza/arac/gun (lambda): 0.0927
Random beklenen cascade orani (24s pencere): 8.85%

=== GOZLEMLENEN vs BEKLENEN ===
Gozlemlenen cascade orani: 14.90%
Beklenen (random):         8.85%
Oran (gozlemlenen/beklenen): x1.68
Z-test: z=49.95, p=0.000000
VERIYE GORE: Cascade orani random beklentinin 1.68x USTUNDE (p<0.05).
  -> Arizalar rastgele degil — gercek bir cascade etkisi var.


---

## 4. Tetikleyici Mekanizma — Kategori × Kategori Matrisi
Hangi sistem arızası hangi sonraki sistemi tetikliyor? Lift skoru + Chi² test.

In [4]:
# BOLUM 4: Tetikleyici Mekanizma — Kategori X Kategori Matrisi
# Soru: Hangi sistem arizasi hangi sonraki arizayi tetikliyor?
# Yontem: Cascade arizalarda onceki_kategori -> ARIZAUSTKODTANIM gecis matrisi + Lift

cascade_df = df[df['cascade'] & df['onceki_kategori'].notna()].copy()
print(f'Cascade ariza kaydi (gecis matrisi icin): {len(cascade_df):,}')

# Gecis matrisi
gecis = cascade_df.groupby(['onceki_kategori', 'ARIZAUSTKODTANIM']).size().reset_index(name='n')

# Lift: P(B sonra | A once) / P(B genel)
# P(B genel) = global frekans
global_freq = df['ARIZAUSTKODTANIM'].value_counts(normalize=True)

# Her A icin: sonraki B'lerin lift skoru
lift_list = []
for a_kat in cascade_df['onceki_kategori'].unique():
    sub = cascade_df[cascade_df['onceki_kategori'] == a_kat]
    if len(sub) < 20: continue
    p_given_a = sub['ARIZAUSTKODTANIM'].value_counts(normalize=True)
    for b_kat, p_a in p_given_a.items():
        p_b = global_freq.get(b_kat, 0)
        if p_b == 0: continue
        lift = p_a / p_b
        n_seq = (sub['ARIZAUSTKODTANIM'] == b_kat).sum()
        if n_seq < 5: continue
        lift_list.append({
            'A_kategori': a_kat,
            'B_kategori': b_kat,
            'n_AB': n_seq,
            'p_B_given_A': p_a,
            'p_B_global': p_b,
            'lift': lift
        })

lift_df = pd.DataFrame(lift_list).sort_values('lift', ascending=False)

print()
print('=== EN GUCLU CASCADE TETIKLEYICILERI (lift > 2) ===')
print('Yorum: Lift=3 -> A oncesi olmasi, B arizasi olasiligini 3x artiriyor')
print()
print(f'{"A (Onceki)":35s}  -> {"B (Sonraki)":35s}  n   P(B|A)%  Lift')
for _, r in lift_df[lift_df['lift'] >= 2].head(25).iterrows():
    print(f'{str(r.A_kategori)[:35]:35s}  -> {str(r.B_kategori)[:35]:35s}  {r.n_AB:3d}  {r.p_B_given_A*100:5.1f}    {r.lift:.2f}')

# Self-cascade (ayni kategori tekrar)
print()
print('=== SELF-CASCADE: Ayni Kategori Tekrar ===')
self_df = lift_df[lift_df['A_kategori'] == lift_df['B_kategori']]
print(self_df[['A_kategori','n_AB','p_B_given_A','lift']].sort_values('lift', ascending=False).head(15).to_string(index=False))

# Cross-cascade (farkli kategori) — top 10
print()
print('=== CROSS-CASCADE: Farkli Kategori (top 10 Lift) ===')
cross_df = lift_df[lift_df['A_kategori'] != lift_df['B_kategori']]
print(cross_df.head(10)[['A_kategori','B_kategori','n_AB','p_B_given_A','lift']].to_string(index=False))

# Chi-square test: gecis matrisi rastgele mi?
print()
print('=== CHI-SQUARE: Gecis Matrisi Rastgele Mi? ===')
# Top 10 en cok onceki kategoriler arasinda Chi²
top_kat = df['ARIZAUSTKODTANIM'].value_counts().head(10).index
top_cascade = cascade_df[cascade_df['onceki_kategori'].isin(top_kat) & cascade_df['ARIZAUSTKODTANIM'].isin(top_kat)]
ct = pd.crosstab(top_cascade['onceki_kategori'], top_cascade['ARIZAUSTKODTANIM'])
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
print(f'Chi²={chi2:.1f}, df={dof}, p={p_chi:.6f}')
if p_chi < 0.05:
    print('VERIYE GORE: Gecis matrisi ANLAMLI sekilde non-random — gercek tetikleyici pattern var.')
else:
    print('VERIYE GORE: Gecis matrisi random ile uyumlu.')


Cascade ariza kaydi (gecis matrisi icin): 8,727

=== EN GUCLU CASCADE TETIKLEYICILERI (lift > 2) ===
Yorum: Lift=3 -> A oncesi olmasi, B arizasi olasiligini 3x artiriyor

A (Onceki)                           -> B (Sonraki)                          n   P(B|A)%  Lift
İLAVE DİREKSİYON SİSTEMİ ARIZALARI   -> İLAVE DİREKSİYON SİSTEMİ ARIZALARI    19   42.2    76.55
ROT AYARLARI                         -> ROT AYARLARI                           8   22.2    33.63
KWS SİSTEMİ ARIZALARI                -> KWS SİSTEMİ ARIZALARI                  9   19.1    33.37
BASINÇLI YAĞ HATTI ARIZASI           -> BASINÇLI YAĞ HATTI ARIZASI            14   24.1    32.72
DİREKSİYON ARIZALARI                 -> DİREKSİYON ARIZALARI                  11   19.0    23.78
BASINÇLI MOTOR YAĞI SİSTEMİ          -> BASINÇLI MOTOR YAĞI SİSTEMİ           12   13.2    13.64
LASTİK ARIZALARI                     -> LASTİK ARIZALARI                       7   11.5    13.25
KAYIŞ KASNAK ARIZALARI               -> KAYIŞ KASNAK AR

---

## 5. Sankey Diyagramı — Görsel Akış
En güçlü cascade tetikleyici→sonraki kategori akışları.

In [5]:
# BOLUM 5: Sankey Diyagrami — Cascade Akis Gorsellestirme

# Top 10 en cok cascade tetikleyen kategori
top_cascade_kat = cascade_df['onceki_kategori'].value_counts().head(10).index.tolist()
top_cascade_kat = list(set(top_cascade_kat + cascade_df['ARIZAUSTKODTANIM'].value_counts().head(10).index.tolist()))

sankey_df = cascade_df[
    cascade_df['onceki_kategori'].isin(top_cascade_kat) &
    cascade_df['ARIZAUSTKODTANIM'].isin(top_cascade_kat)
].groupby(['onceki_kategori', 'ARIZAUSTKODTANIM']).size().reset_index(name='value')

# Min cizgi: 30 kayit
sankey_df = sankey_df[sankey_df['value'] >= 30]

# Source/target/value list
nodes = list(set(sankey_df['onceki_kategori']) | set(sankey_df['ARIZAUSTKODTANIM']))
node_map = {n: i for i, n in enumerate(nodes)}

# Source ve target gostermek icin: source nodelara "_A" target nodelara "_B" eklensin
source_nodes = [k + ' (A)' for k in sankey_df['onceki_kategori'].unique()]
target_nodes = [k + ' (B)' for k in sankey_df['ARIZAUSTKODTANIM'].unique()]
all_nodes = list(set(source_nodes + target_nodes))
n_map = {n: i for i, n in enumerate(all_nodes)}

sources = [n_map[a + ' (A)'] for a in sankey_df['onceki_kategori']]
targets = [n_map[b + ' (B)'] for b in sankey_df['ARIZAUSTKODTANIM']]
values  = sankey_df['value'].tolist()

fig = go.Figure(data=[go.Sankey(
    node=dict(label=all_nodes, pad=15, thickness=20),
    link=dict(source=sources, target=targets, value=values)
)])
fig.update_layout(title='Cascade Arizalar: Tetikleyici (A) -> Sonraki (B)', height=600)
fig.show()

print(f'Sankey: {len(all_nodes)} node, {len(sources)} edge')
print(f'En guclu 5 akis:')
for _, r in sankey_df.nlargest(5, 'value').iterrows():
    print(f'  {r.onceki_kategori[:30]} -> {r.ARIZAUSTKODTANIM[:30]}: {r.value}')


Sankey: 20 node, 60 edge
En guclu 5 akis:
  SOĞUTMA SİSTEMİ ARIZASI -> SOĞUTMA SİSTEMİ ARIZASI: 457
  KAPI ARIZALARI -> KAPI ARIZALARI: 456
  MOTOR ARIZALARI -> MOTOR ARIZALARI: 343
  KLİMA SİSTEMİ ARIZALARI -> KLİMA SİSTEMİ ARIZALARI: 311
  ELEKTRİK SİSTEMİ ARIZALARI -> ELEKTRİK SİSTEMİ ARIZALARI: 305


---

## 6. Arıza Fırtınası — Yoğun Cluster'lar
24s/7g/30g içinde N+ arıza yapan araçların prevalansı + ciddiyet karşılaştırması.

In [6]:
# BOLUM 6: Ariza Firtinasi — Yogun Cluster'lar
# Sorular:
# - 24 saat / 7 gun icinde 3+ ariza yapan kac arac var?
# - Firtina yasayan araclar daha mi ciddi sorunlu?

def firtina_say(df, pencere_saat, esik_n):
    """Her arac icin: zaman penceresinde esik N kez ariza yapma sayisi."""
    sonuc = {}
    for kapino, sub in df.groupby('KAPINO'):
        tarihler = sub['OLAYTARIHI'].sort_values().values
        firtina_n = 0
        for i in range(len(tarihler)):
            pencere = tarihler[(tarihler >= tarihler[i]) &
                                (tarihler <= tarihler[i] + np.timedelta64(pencere_saat, 'h'))]
            if len(pencere) >= esik_n:
                firtina_n += 1
                # Bir kez sayilirsa skip (overlapping pencerelerden kacin)
                break
        sonuc[kapino] = firtina_n
    return sonuc

# 24 saat icinde 3+ ariza
firtina_24s_3 = firtina_say(df, 24, 3)
n_arac_24s3 = sum(1 for v in firtina_24s_3.values() if v >= 1)
print(f'24s icinde 3+ ariza yapan arac: {n_arac_24s3:,} / {df["KAPINO"].nunique():,} (%{n_arac_24s3/df["KAPINO"].nunique()*100:.1f})')

# 7 gun icinde 5+ ariza
firtina_7g_5 = firtina_say(df, 168, 5)
n_arac_7g5 = sum(1 for v in firtina_7g_5.values() if v >= 1)
print(f'7g icinde 5+ ariza yapan arac:  {n_arac_7g5:,} / {df["KAPINO"].nunique():,} (%{n_arac_7g5/df["KAPINO"].nunique()*100:.1f})')

# 30 gun icinde 10+ ariza
firtina_30g_10 = firtina_say(df, 720, 10)
n_arac_30g10 = sum(1 for v in firtina_30g_10.values() if v >= 1)
print(f'30g icinde 10+ ariza yapan arac: {n_arac_30g10:,} / {df["KAPINO"].nunique():,} (%{n_arac_30g10/df["KAPINO"].nunique()*100:.1f})')

# Firtina yasayan vs yasamayan ciddiyet karsilastirmasi
print()
print('=== FIRTINA YASAYAN ARACLAR vs DIGERLERI ===')
firtina_arac = set(k for k, v in firtina_24s_3.items() if v >= 1)
arac_skor = df.groupby('KAPINO')['ciddiyet_skoru'].agg(['mean', 'count']).reset_index()
arac_skor['firtinali'] = arac_skor['KAPINO'].isin(firtina_arac)
arac_skor = arac_skor[arac_skor['count'] >= 5]

firt = arac_skor[arac_skor['firtinali']]['mean']
nfirt = arac_skor[~arac_skor['firtinali']]['mean']

print(f'Firtinali (n={len(firt)}):    ort_skor={firt.mean():.3f}, std={firt.std():.3f}')
print(f'Firtinasiz (n={len(nfirt)}):  ort_skor={nfirt.mean():.3f}, std={nfirt.std():.3f}')

u, p = stats.mannwhitneyu(firt, nfirt, alternative='two-sided')
print(f'Mann-Whitney U: u={u:.0f}, p={p:.6f}')

if p < 0.05:
    fark = firt.mean() - nfirt.mean()
    yon = 'YUKSEK' if fark > 0 else 'DUSUK'
    print(f'VERIYE GORE: Firtinali araclar {yon} ciddiyet skoru (fark: {fark:+.3f}, p<0.05).')
else:
    print(f'VERIYE GORE: Iki grup arasinda fark anlamsiz (p={p:.4f}).')


24s icinde 3+ ariza yapan arac: 613 / 3,509 (%17.5)
7g icinde 5+ ariza yapan arac:  670 / 3,509 (%19.1)
30g icinde 10+ ariza yapan arac: 561 / 3,509 (%16.0)

=== FIRTINA YASAYAN ARACLAR vs DIGERLERI ===
Firtinali (n=613):    ort_skor=3.597, std=0.531
Firtinasiz (n=2703):  ort_skor=3.590, std=0.704
Mann-Whitney U: u=828008, p=0.982833
VERIYE GORE: Iki grup arasinda fark anlamsiz (p=0.9828).


---

## 7. Inter-arrival Time Dağılımı (Survival)
Bir arızadan sonra kaç gün arızasız kalıyor araç? Exponential dağılım uyumu testi.

In [7]:
# BOLUM 7: Inter-arrival Time Dagilimi (Survival)
# Soru: Bir arizadan sonra kac gun arizasiz kaliyor arac?
# Yontem: Kaplan-Meier benzeri ampirik survival fonksiyonu

# Tum ardisik ariza ciftleri
inter = df['saat_farki'].dropna() / 24  # gun cinsinden

print('=== INTER-ARRIVAL TIME (GUN) ===')
print(f'Toplam ardisik kayit: {len(inter):,}')
print(f'Median: {inter.median():.2f} gun')
print(f'Mean:   {inter.mean():.2f} gun')
print(f'P25:    {inter.quantile(0.25):.2f} gun')
print(f'P75:    {inter.quantile(0.75):.2f} gun')
print(f'P95:    {inter.quantile(0.95):.2f} gun')

# Survival fonksiyonu: S(t) = P(T > t)
gunler = np.arange(0, 31, 1)
survival = [(inter > t).mean() for t in gunler]

print()
print('=== SURVIVAL FONKSIYONU: P(Sonraki ariza > t gun) ===')
print(f'{"Gun":5s}  S(t)    Pct ariza t gun icinde')
for t, s in zip(gunler, survival):
    if t in [0, 1, 3, 7, 14, 21, 30]:
        print(f'{t:5d}  {s:.3f}    {(1-s)*100:5.1f}%')

# Exponential mi? Eger oyle ise log-S(t) lineer olmali
# Test: log-likelihood ratio
from scipy.stats import expon
# Exponential parametre tahmini
loc, scale = expon.fit(inter, floc=0)
# K-S test
ks_stat, ks_p = stats.kstest(inter, 'expon', args=(loc, scale))
print()
print('=== DAGILIM TESTI ===')
print(f'Exponential scale tahmini: {scale:.2f} gun')
print(f'K-S test: stat={ks_stat:.4f}, p={ks_p:.6f}')
if ks_p < 0.05:
    print('VERIYE GORE: Inter-arrival dagilimi exponential ile UYUMLU DEGIL (p<0.05).')
    print('  -> Arizalar memoryless degil — gecmis pattern var.')
else:
    print('VERIYE GORE: Inter-arrival exponential ile uyumlu (random Poisson process).')

# Gorsel
sub = inter[inter <= 30]
fig = px.histogram(sub, nbins=30, title='Inter-arrival Time Dagilimi (30 gun)',
                    labels={'value': 'Gun farki'})
fig.show()


=== INTER-ARRIVAL TIME (GUN) ===
Toplam ardisik kayit: 55,050
Median: 5.14 gun
Mean:   9.37 gun
P25:    1.85 gun
P75:    11.97 gun
P95:    32.94 gun

=== SURVIVAL FONKSIYONU: P(Sonraki ariza > t gun) ===
Gun    S(t)    Pct ariza t gun icinde
    0  1.000      0.0%
    1  0.841     15.9%
    3  0.644     35.6%
    7  0.408     59.2%
   14  0.208     79.2%
   21  0.117     88.3%
   30  0.061     93.9%

=== DAGILIM TESTI ===
Exponential scale tahmini: 9.37 gun
K-S test: stat=0.0860, p=0.000000
VERIYE GORE: Inter-arrival dagilimi exponential ile UYUMLU DEGIL (p<0.05).
  -> Arizalar memoryless degil — gecmis pattern var.


---

## 8. Araç Bazlı Cascade Profili
Her araç için cascade oranı + yaş ile korelasyon + bant analizi.

In [8]:
# BOLUM 8: Arac Bazli Cascade Profili
# Her arac icin cascade orani + yas/sefer ile iliski

arac_cascade = df.groupby('KAPINO').agg(
    toplam_ariza = ('cascade', 'count'),
    cascade_n    = ('cascade', 'sum'),
    cascade_orani = ('cascade', 'mean'),
    modelyili    = ('MODELYILI', lambda x: x.mode().iloc[0]),
    ort_skor     = ('ciddiyet_skoru', 'mean'),
    ciddi_oran   = ('ciddi_ariza', 'mean'),
).reset_index()
arac_cascade['arac_yasi'] = 2025 - arac_cascade['modelyili']
arac_cascade = arac_cascade[arac_cascade['toplam_ariza'] >= 5]  # gurultu eleme

print(f'Analiz seti: {len(arac_cascade):,} arac (min 5 ariza)')
print()
print('=== CASCADE ORAN DAGILIMI ===')
print(arac_cascade['cascade_orani'].describe([0.1, 0.25, 0.5, 0.75, 0.9]).round(3).to_string())

# YAS x CASCADE
r_yas, p_yas = stats.pearsonr(arac_cascade['arac_yasi'], arac_cascade['cascade_orani'])
rs_yas, ps_yas = stats.spearmanr(arac_cascade['arac_yasi'], arac_cascade['cascade_orani'])
print()
print('=== YAS X CASCADE_ORAN KORELASYON ===')
print(f'Pearson  r={r_yas:+.3f}  p={p_yas:.4f}')
print(f'Spearman r={rs_yas:+.3f}  p={ps_yas:.4f}')

if p_yas < 0.05:
    yon = 'POZITIF' if r_yas > 0 else 'NEGATIF'
    print(f'VERIYE GORE: Yas ile cascade orani {yon} iliskili.')
else:
    print('VERIYE GORE: Yas ile cascade orani anlamli iliskisiz.')

# Yas bandlari
arac_cascade['yas_bant'] = pd.cut(arac_cascade['arac_yasi'], bins=[0,5,10,15,25],
    labels=['Yeni (0-5)', 'Genc (6-10)', 'Orta (11-15)', 'Yasli (16+)'])
print()
print('=== YAS BANT X CASCADE ===')
print(arac_cascade.groupby('yas_bant', observed=False).agg(
    n_arac=('KAPINO','count'),
    ort_cascade=('cascade_orani','mean'),
    ort_skor=('ort_skor','mean'),
).round(3).to_string())

# CASCADE ORAN BAND X CIDDIYET
print()
print('=== CASCADE ORAN BANT X CIDDIYET SKORU ===')
arac_cascade['cas_bant'] = pd.cut(arac_cascade['cascade_orani'],
    bins=[-0.01, 0.2, 0.4, 0.6, 1.01],
    labels=['Dusuk (<20%)', 'Orta (20-40%)', 'Yuksek (40-60%)', 'Cok Yuksek (60%+)'])
print(arac_cascade.groupby('cas_bant', observed=False).agg(
    n=('KAPINO','count'),
    ort_skor=('ort_skor','mean'),
    ort_ciddi=('ciddi_oran','mean'),
).round(3).to_string())

# ANOVA
gruplar = [g['ort_skor'].values for _, g in arac_cascade.groupby('cas_bant', observed=False) if len(g) >= 5]
if len(gruplar) >= 2:
    f, p_anova = stats.f_oneway(*gruplar)
    print()
    print(f'ANOVA F={f:.2f}, p={p_anova:.6f}')
    if p_anova < 0.05:
        print('VERIYE GORE: Cascade orani bandlari arasinda ciddiyet skoru ANLAMLI farkli.')
    else:
        print('VERIYE GORE: Bandlar arasi fark anlamsiz.')


Analiz seti: 3,316 arac (min 5 ariza)

=== CASCADE ORAN DAGILIMI ===
count    3316.000
mean        0.125
std         0.095
min         0.000
10%         0.000
25%         0.053
50%         0.125
75%         0.190
90%         0.250
max         0.476

=== YAS X CASCADE_ORAN KORELASYON ===
Pearson  r=+0.030  p=0.0843
Spearman r=+0.037  p=0.0313
VERIYE GORE: Yas ile cascade orani anlamli iliskisiz.

=== YAS BANT X CASCADE ===
              n_arac  ort_cascade  ort_skor
yas_bant                                   
Yeni (0-5)       331        0.131     3.436
Genc (6-10)      519        0.111     3.506
Orta (11-15)    1803        0.126     3.604
Yasli (16+)      663        0.130     3.702

=== CASCADE ORAN BANT X CIDDIYET SKORU ===
                      n  ort_skor  ort_ciddi
cas_bant                                    
Dusuk (<20%)       2636     3.597      0.368
Orta (20-40%)       669     3.569      0.375
Yuksek (40-60%)      11     3.593      0.356
Cok Yuksek (60%+)     0       NaN        

---

## 9. Confounder Kontrolü (Multiple Regression)
Cascade-ciddiyet ilişkisi yaş/sefer/garaj kontrolü altında hâlâ anlamlı mı?

In [9]:
# BOLUM 9: Confounder Kontrolu
# Soru: Cascade orani ile ciddiyet iliskisi yas/sefer gibi confounder'lardan mi geliyor?

# Sefer sayisini ekle (arac_gunluk_hatlar.csv'den)
arac_hatlar = pd.read_csv('../panel_data/temiz_veri/arac_gunluk_hatlar.csv')
SEFER_KOL = [c for c in arac_hatlar.columns if 'SEFER' in c.upper()][0]
arac_sefer = arac_hatlar.groupby('KAPINO')[SEFER_KOL].sum().reset_index()
arac_sefer.columns = ['KAPINO', 'toplam_sefer']

arac_cas_full = arac_cascade.merge(arac_sefer, on='KAPINO', how='left')

# GARAJ
arac_garaj = df.groupby('KAPINO')['GARAJ'].agg(lambda x: x.mode().iloc[0]).reset_index()
arac_cas_full = arac_cas_full.merge(arac_garaj, on='KAPINO', how='left')

# 1. SEFER X CASCADE
r_s, p_s = stats.pearsonr(arac_cas_full['toplam_sefer'].fillna(0), arac_cas_full['cascade_orani'])
print('=== CONFOUNDER 1: SEFER YOGUNLUGU ===')
print(f'Sefer x Cascade_oran: r={r_s:+.3f}, p={p_s:.4f}')
if p_s < 0.05:
    yon = 'POZITIF' if r_s > 0 else 'NEGATIF'
    print(f'  -> {yon} iliski: cok sefer = {"cok" if r_s>0 else "az"} cascade.')

# 2. PARTIAL CORRELATION: yas+sefer kontrol altinda cascade-skor iliskisi
import statsmodels.api as sm
from statsmodels.formula.api import ols

print()
print('=== MULTIPLE REGRESSION: ort_skor ~ cascade_orani + yas + sefer + garaj ===')
regdf = arac_cas_full.dropna(subset=['toplam_sefer', 'arac_yasi', 'GARAJ'])
regdf['log_sefer'] = np.log1p(regdf['toplam_sefer'])

m1 = ols('ort_skor ~ cascade_orani', data=regdf).fit()
m2 = ols('ort_skor ~ cascade_orani + arac_yasi + log_sefer', data=regdf).fit()
m3 = ols('ort_skor ~ cascade_orani + arac_yasi + log_sefer + C(GARAJ)', data=regdf).fit()

print()
print(f'{"Model":42s}  cascade_katsayi  p_cascade  R^2')
for m, ad in [(m1,'M1: sadece cascade'), (m2,'M2: + yas + log_sefer'), (m3,'M3: + garaj')]:
    k = m.params['cascade_orani']
    p = m.pvalues['cascade_orani']
    print(f'{ad:42s}  {k:+.4f}          {p:.6f}  {m.rsquared:.3f}')

# Yorum
p3 = m3.pvalues['cascade_orani']
k3 = m3.params['cascade_orani']
print()
if p3 < 0.05:
    if k3 > 0:
        print(f'VERIYE GORE: Confounder kontrolu altinda cascade_orani POZITIF ve anlamli (p={p3:.4f}).')
        print(f'  -> Cascade orani 0 -> 1 = {k3:+.3f} ciddiyet skoru artisi.')
        print('  -> "Cascade = sorunlu arac" iliskisi VERIYLE TEYIT.')
    else:
        print(f'VERIYE GORE: Confounder altinda cascade NEGATIF anlamli (p={p3:.4f}).')
        print('  -> Beklenmedik ters yon — paradoks.')
else:
    print(f'VERIYE GORE: Confounder altinda cascade etkisi ANLAMSIZ (p={p3:.4f}).')
    print('  -> Cascade-skor iliskisi buyuk olcude yas/sefer/garaj farkindan geliyor.')


=== CONFOUNDER 1: SEFER YOGUNLUGU ===
Sefer x Cascade_oran: r=-0.141, p=0.0000
  -> NEGATIF iliski: cok sefer = az cascade.

=== MULTIPLE REGRESSION: ort_skor ~ cascade_orani + yas + sefer + garaj ===

Model                                       cascade_katsayi  p_cascade  R^2
M1: sadece cascade                          -0.0375          0.762474  0.000
M2: + yas + log_sefer                       -0.1859          0.131430  0.033
M3: + garaj                                 -0.3640          0.002494  0.210

VERIYE GORE: Confounder altinda cascade NEGATIF anlamli (p=0.0025).
  -> Beklenmedik ters yon — paradoks.


---

## 10. ML Feature Türetme
12 yeni feature kandidatı (zaman pencereli + cascade tabanlı + tetik lift).

In [10]:
# BOLUM 10: ML FEATURE TURETME
# Cascade analizinden ML model icin guclu feature kandidatlari uret.

print('=== ML FEATURE TURETME ===')
print('Her ariza icin t = ariza zamani olmak uzere "o ana kadar olan" gecmis feature\'lari.')
print()

# Her ariza icin: o ariza ZAMANI ICIN onceki feature'lar
# (Bu, gercek tahmin senaryosunu simule eder — gelecegi bilmiyoruz)

df_feat = df.copy()
df_feat = df_feat.sort_values(['KAPINO', 'OLAYTARIHI']).reset_index(drop=True)

# Cumulative features (her arac icin gecmis say)
df_feat['ariza_sira']      = df_feat.groupby('KAPINO').cumcount()
df_feat['gecmis_ciddi_n']  = df_feat.groupby('KAPINO')['ciddi_ariza'].cumsum().shift(1).fillna(0)
df_feat['gecmis_ariza_n']  = df_feat['ariza_sira']

# Time-window features (rolling)
def gecmis_pencere(df_arac, gun):
    """Bu arac icin: her ariza icin son N gun arıza sayisi."""
    sonuc = []
    tarihler = df_arac['OLAYTARIHI'].values
    for i, t in enumerate(tarihler):
        n = ((tarihler >= t - np.timedelta64(gun, 'D')) & (tarihler < t)).sum()
        sonuc.append(n)
    return sonuc

print('Pencere bazli feature\'lar hesaplaniyor...')
df_feat['ariza_son_7g']   = 0
df_feat['ariza_son_30g']  = 0
df_feat['ariza_son_60g']  = 0
df_feat['ciddi_son_30g']  = 0
df_feat['ciddi_son_7g']   = 0

for kapino, sub in df_feat.groupby('KAPINO'):
    idx = sub.index
    tarihler = sub['OLAYTARIHI'].values
    ciddi    = sub['ciddi_ariza'].values
    for i, t in enumerate(tarihler):
        mask_30 = (tarihler >= t - np.timedelta64(30, 'D')) & (tarihler < t)
        mask_7  = (tarihler >= t - np.timedelta64(7,  'D')) & (tarihler < t)
        mask_60 = (tarihler >= t - np.timedelta64(60, 'D')) & (tarihler < t)

        df_feat.at[idx[i], 'ariza_son_7g']  = mask_7.sum()
        df_feat.at[idx[i], 'ariza_son_30g'] = mask_30.sum()
        df_feat.at[idx[i], 'ariza_son_60g'] = mask_60.sum()
        df_feat.at[idx[i], 'ciddi_son_30g'] = ciddi[mask_30].sum()
        df_feat.at[idx[i], 'ciddi_son_7g']  = ciddi[mask_7].sum()

# Son ariza/ciddi ariza zamani
df_feat['son_ariza_gun'] = df_feat.groupby('KAPINO')['gun_farki'].shift(0).fillna(999)

# Farkli kategori sayisi son 30g
def farkli_kat_30g(df_arac):
    sonuc = []
    tarihler = df_arac['OLAYTARIHI'].values
    kategoriler = df_arac['ARIZAUSTKODTANIM'].values
    for i, t in enumerate(tarihler):
        mask = (tarihler >= t - np.timedelta64(30, 'D')) & (tarihler < t)
        sonuc.append(len(np.unique(kategoriler[mask])) if mask.any() else 0)
    return sonuc

print('Farkli kategori sayisi hesaplaniyor...')
df_feat['farkli_kat_30g'] = 0
for kapino, sub in df_feat.groupby('KAPINO'):
    idx = sub.index
    vals = farkli_kat_30g(sub)
    for j, v in enumerate(vals):
        df_feat.at[idx[j], 'farkli_kat_30g'] = v

# Cascade riski (binary): son 24s veya son 7g icinde 2+ ariza
df_feat['cascade_24s'] = (df_feat['saat_farki'] < 24).fillna(False).astype(int)
df_feat['cascade_7g_3plus'] = (df_feat['ariza_son_7g'] >= 3).astype(int)

# Tetikleyici kategori riski (lift)
# Onceki kategorinin global lift skoru
kategori_lift = lift_df.groupby('A_kategori')['lift'].max().to_dict()
df_feat['tetik_kat_lift'] = df_feat['onceki_kategori'].map(kategori_lift).fillna(1.0)

print()
print(f'Feature seti hazir: {len(df_feat):,} satir')
print('Yeni feature\'lar:')
features = ['gecmis_ariza_n','gecmis_ciddi_n','ariza_son_7g','ariza_son_30g',
            'ariza_son_60g','ciddi_son_30g','ciddi_son_7g',
            'son_ariza_gun','farkli_kat_30g','cascade_24s','cascade_7g_3plus','tetik_kat_lift']
for f in features:
    print(f'  {f}')


=== ML FEATURE TURETME ===
Her ariza icin t = ariza zamani olmak uzere "o ana kadar olan" gecmis feature'lari.

Pencere bazli feature'lar hesaplaniyor...
Farkli kategori sayisi hesaplaniyor...

Feature seti hazir: 58,559 satir
Yeni feature'lar:
  gecmis_ariza_n
  gecmis_ciddi_n
  ariza_son_7g
  ariza_son_30g
  ariza_son_60g
  ciddi_son_30g
  ciddi_son_7g
  son_ariza_gun
  farkli_kat_30g
  cascade_24s
  cascade_7g_3plus
  tetik_kat_lift


---

## 11. Feature × Ciddiyet Korelasyon Testleri
Her feature için Pearson + Spearman + point-biserial.

In [11]:
# BOLUM 11: Feature Korelasyon Testleri
# Her feature x ciddiyet_skoru + ciddi_ariza icin Pearson + Spearman

print('=== FEATURE x CIDDIYET KORELASYON ===')
print(f'{"Feature":25s}  Pearson_r  p        Spearman_rs  p')

features = ['gecmis_ariza_n','gecmis_ciddi_n','ariza_son_7g','ariza_son_30g',
            'ariza_son_60g','ciddi_son_30g','ciddi_son_7g',
            'son_ariza_gun','farkli_kat_30g','cascade_24s','cascade_7g_3plus','tetik_kat_lift']

sonuclar_korr = []
for f in features:
    valid = df_feat.dropna(subset=[f])
    r, p = stats.pearsonr(valid[f], valid['ciddiyet_skoru'])
    rs, ps = stats.spearmanr(valid[f], valid['ciddiyet_skoru'])
    sonuclar_korr.append({'feature': f, 'r': r, 'p': p, 'rs': rs, 'ps': ps, 'abs_r': abs(r)})
    print(f'{f:25s}  {r:+.4f}    {p:.4f}   {rs:+.4f}      {ps:.4f}')

korr_df = pd.DataFrame(sonuclar_korr).sort_values('abs_r', ascending=False)

print()
print('=== EN GUCLU FEATURE\'LAR (|r| sirali) ===')
for _, row in korr_df.head(8).iterrows():
    yon = '✓' if row['r'] > 0 else '✗'
    print(f'{yon} {row.feature:25s}: |r|={row.abs_r:.3f}  yon={"+" if row.r>0 else "-"}')

# Ciddi_ariza ile point-biserial korelasyon
print()
print('=== FEATURE x CIDDI_ARIZA (binary) ===')
for f in features:
    valid = df_feat.dropna(subset=[f])
    r, p = stats.pointbiserialr(valid['ciddi_ariza'], valid[f])
    print(f'{f:25s}  r={r:+.4f}  p={p:.4f}')

# Karar
print()
print('=== ML EKLENEBILIR FEATURE\'LAR (|r| > 0.10) ===')
guclu = korr_df[korr_df['abs_r'] > 0.10]
if len(guclu) > 0:
    for _, r in guclu.iterrows():
        print(f'  {r.feature}  (r={r.r:+.3f})')
else:
    print('  Hicbir feature |r|>0.10 — tum sinyaller zayif.')


=== FEATURE x CIDDIYET KORELASYON ===
Feature                    Pearson_r  p        Spearman_rs  p
gecmis_ariza_n             +0.0277    0.0000   +0.0508      0.0000
gecmis_ciddi_n             +0.0492    0.0000   +0.0741      0.0000
ariza_son_7g               +0.0368    0.0000   +0.0526      0.0000
ariza_son_30g              +0.0280    0.0000   +0.0500      0.0000
ariza_son_60g              +0.0332    0.0000   +0.0555      0.0000
ciddi_son_30g              +0.0611    0.0000   +0.0821      0.0000
ciddi_son_7g               +0.0634    0.0000   +0.0739      0.0000
son_ariza_gun              -0.0208    0.0000   -0.0590      0.0000
farkli_kat_30g             +0.0313    0.0000   +0.0491      0.0000
cascade_24s                +0.0378    0.0000   +0.0427      0.0000
cascade_7g_3plus           +0.0249    0.0000   +0.0340      0.0000
tetik_kat_lift             +0.0047    0.2526   +0.0005      0.8963

=== EN GUCLU FEATURE'LAR (|r| sirali) ===
✓ ciddi_son_7g             : |r|=0.063  yon=+
✓ ciddi

---

## 12. Bant Analizi — En Güçlü Feature'lar
Feature bantlarında ciddiyet skoru ortalaması (ANOVA).

In [12]:
# BOLUM 12: Bant Analizi — En Guclu Feature'lar Icin
# En guclu feature'larin bantlarinda ciddiyet skoru ortalamasi ANOVA ile

print('=== BANT ANALIZI: gecmis_ciddi_n ===')
df_feat['gciddi_bant'] = pd.cut(df_feat['gecmis_ciddi_n'],
    bins=[-0.1, 0.5, 2.5, 5.5, 100],
    labels=['Hic (0)', 'Az (1-2)', 'Orta (3-5)', 'Cok (6+)'])
bant1 = df_feat.groupby('gciddi_bant', observed=False)['ciddiyet_skoru'].agg(['count','mean','std']).round(3)
print(bant1.to_string())

gruplar = [g['ciddiyet_skoru'].values for _, g in df_feat.groupby('gciddi_bant', observed=False) if len(g) >= 10]
if len(gruplar) >= 2:
    f, p = stats.f_oneway(*gruplar)
    print(f'\nANOVA F={f:.2f}, p={p:.6f}')
    if p < 0.05:
        print('VERIYE GORE: gecmis_ciddi_n bandlari arasinda ciddi skor ANLAMLI farkli.')

print()
print('=== BANT ANALIZI: ariza_son_30g ===')
df_feat['s30g_bant'] = pd.cut(df_feat['ariza_son_30g'],
    bins=[-0.1, 0.5, 2.5, 5.5, 100],
    labels=['Hic', '1-2', '3-5', '6+'])
bant2 = df_feat.groupby('s30g_bant', observed=False)['ciddiyet_skoru'].agg(['count','mean','std']).round(3)
print(bant2.to_string())

gruplar = [g['ciddiyet_skoru'].values for _, g in df_feat.groupby('s30g_bant', observed=False) if len(g) >= 10]
if len(gruplar) >= 2:
    f, p = stats.f_oneway(*gruplar)
    print(f'\nANOVA F={f:.2f}, p={p:.6f}')

print()
print('=== BANT ANALIZI: farkli_kat_30g ===')
df_feat['fk30g_bant'] = pd.cut(df_feat['farkli_kat_30g'],
    bins=[-0.1, 0.5, 2.5, 5.5, 100],
    labels=['0', '1-2', '3-5', '6+'])
bant3 = df_feat.groupby('fk30g_bant', observed=False)['ciddiyet_skoru'].agg(['count','mean','std']).round(3)
print(bant3.to_string())

# son_ariza_gun analizi
print()
print('=== BANT ANALIZI: son_ariza_gun (kucuk = yakın zaman) ===')
df_feat['sag_bant'] = pd.cut(df_feat['son_ariza_gun'],
    bins=[-0.1, 1, 7, 30, 1000],
    labels=['<1g', '1-7g', '7-30g', '>30g/hic'])
bant4 = df_feat.groupby('sag_bant', observed=False)['ciddiyet_skoru'].agg(['count','mean','std']).round(3)
print(bant4.to_string())
gruplar = [g['ciddiyet_skoru'].values for _, g in df_feat.groupby('sag_bant', observed=False) if len(g) >= 10]
if len(gruplar) >= 2:
    f, p = stats.f_oneway(*gruplar)
    print(f'\nANOVA F={f:.2f}, p={p:.6f}')


=== BANT ANALIZI: gecmis_ciddi_n ===
             count   mean    std
gciddi_bant                     
Hic (0)       7097  3.370  1.918
Az (1-2)     16749  3.560  1.924
Orta (3-5)   16900  3.650  1.854
Cok (6+)     17813  3.714  1.766

ANOVA F=65.05, p=0.000000
VERIYE GORE: gecmis_ciddi_n bandlari arasinda ciddi skor ANLAMLI farkli.

=== BANT ANALIZI: ariza_son_30g ===
           count   mean    std
s30g_bant                     
Hic         6884  3.499  1.978
1-2        18548  3.576  1.926
3-5        21622  3.630  1.821
6+         11505  3.693  1.738

ANOVA F=18.81, p=0.000000

=== BANT ANALIZI: farkli_kat_30g ===
            count   mean    std
fk30g_bant                     
0            6884  3.499  1.978
1-2         24299  3.587  1.899
3-5         24515  3.649  1.793
6+           2861  3.742  1.758

=== BANT ANALIZI: son_ariza_gun (kucuk = yakın zaman) ===
          count   mean    std
sag_bant                     
<1g        8727  3.778  1.851
1-7g      23888  3.627  1.807
7-30g 

---

## 13. Vaka Analizi — En Çok Cascade Yapan 10 Araç
Spesifik araç profili, geçiş pattern'ı, zaman çizelgesi.

In [13]:
# BOLUM 13: Vaka Analizi — En Cok Cascade Yapan 10 Arac

# Cascade orani en yuksek araclar (min 10 ariza)
en_cascade = arac_cascade[arac_cascade['toplam_ariza'] >= 10].nlargest(10, 'cascade_orani')

print('=== EN COK CASCADE YAPAN 10 ARAC ===')
print(f'{"KAPINO":10s}  yas  toplam_n  cascade_n  cascade_%  ort_skor')
for _, r in en_cascade.iterrows():
    print(f'{r.KAPINO:10s}  {r.arac_yasi:3.0f}  {r.toplam_ariza:8d}  {r.cascade_n:9d}  {r.cascade_orani*100:7.1f}%  {r.ort_skor:.2f}')

# Bu araclar icin tipik cascade akisi nedir?
print()
print('=== BU ARACLARDA EN COK GORULEN CASCADE GECISLERI ===')
high_cas = set(en_cascade['KAPINO'])
high_cas_df = cascade_df[cascade_df['KAPINO'].isin(high_cas)]
gecisler = high_cas_df.groupby(['onceki_kategori', 'ARIZAUSTKODTANIM']).size().reset_index(name='n')
gecisler = gecisler.sort_values('n', ascending=False).head(10)
for _, r in gecisler.iterrows():
    print(f'  {r.onceki_kategori[:25]} -> {r.ARIZAUSTKODTANIM[:25]}: {r.n}')

# Zaman cizelgesi: ilk arac
ilk_arac = en_cascade.iloc[0]['KAPINO']
print()
print(f'=== {ilk_arac} ZAMAN CIZELGESI (ornek) ===')
ilk_data = df[df['KAPINO'] == ilk_arac].sort_values('OLAYTARIHI')
print(f'Toplam ariza: {len(ilk_data)}')
print(f'Ilk ariza:    {ilk_data["OLAYTARIHI"].min().strftime("%Y-%m-%d %H:%M")}')
print(f'Son ariza:    {ilk_data["OLAYTARIHI"].max().strftime("%Y-%m-%d %H:%M")}')

# Cascade zincirleri (24s icinde tekrar tekrar)
zincir_n = 0
mevcut_zincir = 1
for i in range(1, len(ilk_data)):
    if ilk_data.iloc[i]['saat_farki'] < 24:
        mevcut_zincir += 1
    else:
        if mevcut_zincir >= 2:
            zincir_n += 1
        mevcut_zincir = 1
if mevcut_zincir >= 2:
    zincir_n += 1
print(f'Cascade zincir sayisi (24s icinde ardisik): {zincir_n}')


=== EN COK CASCADE YAPAN 10 ARAC ===
KAPINO      yas  toplam_n  cascade_n  cascade_%  ort_skor
K3729        12        21         10     47.6%  3.44
M6100        13        20          9     45.0%  2.42
K5729        12        23         10     43.5%  3.14
M2185        19        19          8     42.1%  3.93
O3643         3        43         18     41.9%  4.13
K4256        12        12          5     41.7%  4.28
M3261        17        24         10     41.7%  3.95
T3715        11        12          5     41.7%  3.51
T4689        11        24         10     41.7%  3.89
O6772         3        22          9     40.9%  3.66

=== BU ARACLARDA EN COK GORULEN CASCADE GECISLERI ===
  KLİMA SİSTEMİ ARIZALARI -> KLİMA SİSTEMİ ARIZALARI: 6
  KAPI ARIZALARI -> KAPI ARIZALARI: 5
  MOTOR ARIZALARI -> ELEKTRİK SİSTEMİ ARIZALAR: 4
  ELEKTRİK SİSTEMİ ARIZALAR -> KAPI ARIZALARI: 4
  ELEKTRİK SİSTEMİ ARIZALAR -> ELEKTRİK SİSTEMİ ARIZALAR: 3
  SOĞUTMA SİSTEMİ ARIZASI -> MOTOR ARIZALARI: 3
  MOTOR ARIZALARI -

---

## 14. Operasyonel Uyarı Eşiği
Hangi pattern flag edilmeli? Lift skoruna göre uyarı kural önerileri.

In [14]:
# BOLUM 14: Operasyonel Uyari Esigi
# Soru: Hangi pattern flag edilmeli? Hangi esik ML/uyari sistemi icin uygun?

# Test edilecek "uyari kurali" senaryolari
print('=== UYARI ESIK SENARYOLARI ===')
print('Kural: bu pattern\'i gosteren arac "yuksek risk" olarak isaretlensin')
print()

senaryolar = [
    ('Son 7g 3+ ariza',        df_feat['ariza_son_7g']  >= 3),
    ('Son 30g 5+ ariza',       df_feat['ariza_son_30g'] >= 5),
    ('Son 7g 2+ ciddi',        df_feat['ciddi_son_7g']  >= 2),
    ('Son 30g 3+ ciddi',       df_feat['ciddi_son_30g'] >= 3),
    ('Son 30g 3+ farkli kat',  df_feat['farkli_kat_30g'] >= 3),
    ('Cascade 24s',            df_feat['cascade_24s']    == 1),
    ('Tetik_lift > 2',         df_feat['tetik_kat_lift'] > 2),
]

print(f'{"Kural":35s}  flag_n  ort_ciddi  ciddi_or_flag  ciddi_or_diger  Lift')
for ad, kosul in senaryolar:
    flag = df_feat[kosul]
    diger = df_feat[~kosul]
    if len(flag) < 50: continue
    ort_skor_f = flag['ciddiyet_skoru'].mean()
    ort_skor_d = diger['ciddiyet_skoru'].mean()
    cf = flag['ciddi_ariza'].mean()
    cd = diger['ciddi_ariza'].mean()
    lift = cf / cd if cd > 0 else 0
    print(f'{ad:35s}  {len(flag):6d}  {ort_skor_f:.3f}     {cf*100:6.1f}%       {cd*100:6.1f}%       {lift:.2f}')

# En guclu uyari kuralini sec
print()
print('=== KARAR ===')
print('Lift > 1.5 olan uyarilar = MUDAHALE EDILMELI (kaza/ciddi ariza onlenmesi)')

# Kombinasyon: AND birlestirme
print()
print('=== KOMBINASYON UYARI ===')
kombi = (df_feat['ariza_son_30g'] >= 3) & (df_feat['farkli_kat_30g'] >= 2)
flag = df_feat[kombi]
if len(flag) >= 50:
    diger = df_feat[~kombi]
    cf = flag['ciddi_ariza'].mean()
    cd = diger['ciddi_ariza'].mean()
    print(f'Kural: son 30g 3+ ariza VE 2+ farkli kategori')
    print(f'  flag: {len(flag):,}, ciddi orani: {cf*100:.1f}%')
    print(f'  diger: {len(diger):,}, ciddi orani: {cd*100:.1f}%')
    print(f'  Lift: {cf/cd:.2f}')


=== UYARI ESIK SENARYOLARI ===
Kural: bu pattern'i gosteren arac "yuksek risk" olarak isaretlensin

Kural                                flag_n  ort_ciddi  ciddi_or_flag  ciddi_or_diger  Lift
Son 7g 3+ ariza                        5420  3.755       43.5%         37.4%       1.16
Son 30g 5+ ariza                      17157  3.671       41.1%         36.7%       1.12
Son 7g 2+ ciddi                        3456  3.914       47.1%         37.4%       1.26
Son 30g 3+ ciddi                      10212  3.782       44.5%         36.6%       1.21
Son 30g 3+ farkli kat                 27376  3.659       40.1%         36.2%       1.11
Cascade 24s                            8727  3.778       43.5%         37.1%       1.17
Tetik_lift > 2                        54657  3.616       38.3%         34.0%       1.13

=== KARAR ===
Lift > 1.5 olan uyarilar = MUDAHALE EDILMELI (kaza/ciddi ariza onlenmesi)

=== KOMBINASYON UYARI ===
Kural: son 30g 3+ ariza VE 2+ farkli kategori
  flag: 32,440, ciddi orani: 4

---

## 15. Alt Kategori (ARIZAKODU) Bazlı Lift
Üst kategori lift zayıftı. Alt sistem (ARIZAKODU) bazında tekrar pattern daha güçlü olabilir.

**DİKKAT:** Bakım türü (değişim vs tamir) verimizde YOK. Sebep yorumlamaz, sadece veri raporlar.

In [15]:
# BOLUM 15: Alt Kategori (ARIZAKODU) Bazli Lift
# Ust kategori lift'i zayifti (r=0.005). Alt sistem (ARIZAKODU) bazinda daha guclu olabilir.
# Veri tabani: bu seviyede tekrar pattern hangi spesifik parcalarda guclu?
#
# DIKKAT: Bakim turu (degisim vs tamir) verimizde YOK. Yani:
# - Ayni ARIZAKODU tekrari = parca degismedi mi, tamir kalitesi dusuk mu, yoksa farkli bir nedenle mi?
# - Bilemeyiz. Sadece veriyi raporlariz.

# Alt kategori (ARIZAKODU) bazinda self-cascade lift
df_ak = df.copy()
df_ak['onceki_ariza_kodu'] = df_ak.groupby('KAPINO')['ARIZAKODU'].shift(1)
df_ak['cascade'] = df_ak['saat_farki'] < 24

cascade_ak = df_ak[df_ak['cascade'] & df_ak['onceki_ariza_kodu'].notna()].copy()
print(f'Alt kategori cascade kayit: {len(cascade_ak):,}')

# Self-cascade: ayni ARIZAKODU tekrar
self_ak = cascade_ak[cascade_ak['ARIZAKODU'] == cascade_ak['onceki_ariza_kodu']]
print(f'Self-cascade (ayni ARIZAKODU): {len(self_ak):,} ({len(self_ak)/len(cascade_ak)*100:.1f}%)')

# Her ARIZAKODU icin tekrar lift'i
ak_n = df_ak['ARIZAKODU'].value_counts()  # genel frekans
ak_global_oran = ak_n / ak_n.sum()

# Cascade icinde ARIZAKODU oranlari
ak_cascade_oran = cascade_ak['ARIZAKODU'].value_counts(normalize=True)

# Self-cascade lift: P(B = onceki_A | cascade) / P(B genel)
ak_lift_list = []
for ak in ak_cascade_oran.index:
    n_self = ((cascade_ak['ARIZAKODU'] == ak) & (cascade_ak['onceki_ariza_kodu'] == ak)).sum()
    if n_self < 5: continue
    n_cascade_total = (cascade_ak['onceki_ariza_kodu'] == ak).sum()
    if n_cascade_total < 10: continue
    p_self_given_a = n_self / n_cascade_total  # P(B=A | A oncesi)
    p_global = ak_global_oran.get(ak, 0)
    if p_global == 0: continue
    self_lift = p_self_given_a / p_global

    # Aciklama icin: bu ARIZAKODU'nun ust kategori adi
    ust_kat = df_ak[df_ak['ARIZAKODU'] == ak]['ARIZAUSTKODTANIM'].mode().iloc[0]
    ak_lift_list.append({
        'arizakodu': ak,
        'ust_kat': ust_kat,
        'n_self': n_self,
        'n_total': n_cascade_total,
        'self_orani': p_self_given_a,
        'global_orani': p_global,
        'self_lift': self_lift,
    })

ak_lift_df = pd.DataFrame(ak_lift_list).sort_values('self_lift', ascending=False)

print()
print('=== EN GUCLU ALT KATEGORI SELF-CASCADE LIFT (top 20) ===')
print(f'{"ARIZAKODU":50s}  ust_kat                 n_self  Lift')
for _, r in ak_lift_df.head(20).iterrows():
    print(f'{str(r.arizakodu)[:50]:50s}  {str(r.ust_kat)[:22]:22s}  {r.n_self:5d}  {r.self_lift:.1f}')

# Yorum
print()
print('=== YORUM ===')
print('Bu liste "veri ne diyor" raporu:')
print(' - Yuksek lift = bu spesifik ARIZAKODU 24s icinde tekrar etme egiliminde yuksek')
print(' - SEBEPLERI bilemeyiz: parca degisimi olmadi mi, tamir kalitesi dusuk mu, yoksa farkli bir')
print('   neden mi?')
print(' - Lift kendi basina karar mekanizmasi degil, sadece raporlanan veri pattern')


Alt kategori cascade kayit: 8,727
Self-cascade (ayni ARIZAKODU): 1,734 (19.9%)

=== EN GUCLU ALT KATEGORI SELF-CASCADE LIFT (top 20) ===
ARIZAKODU                                           ust_kat                 n_self  Lift
VİTESTEN ATIYOR                                     OTOMATİK ŞANZIMAN ARIZ      9  136.0
KORNA ÇALMIYOR                                      ELEKTRİK SİSTEMİ ARIZA      8  110.9
İÇ AYDINLATMA LAMBALARI YANMIYOR                    ELEKTRİK SİSTEMİ ARIZA      9  96.1
KAMERA AÇILARI UYGUN DEĞİL                          KAMERA SİSTEMİ ARIZALA      6  96.0
ŞOFÖR KABİN KAPISI ARIZALI                          KAROSER ARIZALARI           5  87.1
KAPI DURACAK DÜĞMESİ ARIZALI                        KAPI ARIZALARI              5  72.7
BEYİN ARIZALARI                                     ELEKTRİK SİSTEMİ ARIZA      9  71.2
AKBİL OKUMUYOR                                      AKBİL ARIZALARI             9  57.8
ŞANZIMAN YAĞ SICAKLIĞI YÜKSEK                       OTOMATİK ŞANZIMA

---

## 16. Cascade × Sistem Subgroup
Hangi sistemde cascade ciddiyet artırıyor, hangisinde değil? Mann-Whitney testleri.

**Veri ne diyor:** Bakım türü bilinmez, sebep yorumlanmaz.

In [16]:
# BOLUM 16: Cascade × Sistem Subgroup — Hangi Sistemlerde Cascade Ciddiyet ARTTIRIYOR?
# Soru: Cascade pattern her sistemde ayni mi? Bazi sistemlerin cascade'i ciddi, bazilari degil mi?
# DIKKAT: Bakim turu (degisim vs tamir) yok. Sebep yorumlanmayacak, sadece veri.

df_feat_x = df_feat.copy()
df_feat_x['kategori'] = df_feat_x['ARIZAUSTKODTANIM']

# Her kategori icin: cascade arizalarinda ortalama ciddiyet skoru vs stand-alone
print('=== KATEGORI BAZLI: CASCADE vs STAND-ALONE CIDDIYET ===')
print(f'{"Kategori":35s}  cascade_n  ort_skor_C  stand_n   ort_skor_S  fark    p')

sistem_sonuc = []
for kat in df_feat_x['kategori'].value_counts().head(20).index:
    sub = df_feat_x[df_feat_x['kategori'] == kat]
    cascade_sub  = sub[sub['cascade_24s'] == 1]
    standalone_sub = sub[sub['cascade_24s'] == 0]
    if len(cascade_sub) < 20 or len(standalone_sub) < 50: continue

    skor_c = cascade_sub['ciddiyet_skoru'].mean()
    skor_s = standalone_sub['ciddiyet_skoru'].mean()
    fark = skor_c - skor_s

    u, p = stats.mannwhitneyu(cascade_sub['ciddiyet_skoru'],
                                standalone_sub['ciddiyet_skoru'],
                                alternative='two-sided')
    sistem_sonuc.append({
        'kategori': kat, 'cascade_n': len(cascade_sub),
        'ort_C': skor_c, 'ort_S': skor_s, 'fark': fark, 'p': p
    })
    isaret = '✓' if p < 0.05 else ' '
    print(f'{isaret} {str(kat)[:34]:34s}  {len(cascade_sub):5d}     {skor_c:.3f}      {len(standalone_sub):5d}    {skor_s:.3f}     {fark:+.3f}  {p:.4f}')

sis_df = pd.DataFrame(sistem_sonuc)

# Hangi sistemlerde cascade gercekten ciddi (pozitif fark + anlamli)?
print()
print('=== CASCADE\'I GERCEKTEN CIDDI OLAN SISTEMLER (fark>0.1 ve p<0.05) ===')
ciddi_cascade = sis_df[(sis_df['fark'] > 0.1) & (sis_df['p'] < 0.05)]
if len(ciddi_cascade) > 0:
    for _, r in ciddi_cascade.iterrows():
        print(f'  {r.kategori[:35]}: cascade ort_skor {r.ort_C:.3f} vs stand {r.ort_S:.3f} (+{r.fark:.3f}, p={r.p:.4f})')
else:
    print('  Hicbir sistem icin cascade ANLAMLI sekilde daha ciddi degil.')

print()
print('=== CASCADE\'I CIDDI OLMAYAN SISTEMLER (fark<=0 veya anlamsiz) ===')
hafif = sis_df[(sis_df['fark'] <= 0) | (sis_df['p'] >= 0.05)]
print(f'Toplam {len(hafif)} sistem - cascade ortalama ciddiyet farkindan oluşturmuyor.')
for _, r in hafif.head(5).iterrows():
    print(f'  {r.kategori[:35]}: fark={r.fark:+.3f}, p={r.p:.4f}')

print()
print('=== YORUM ===')
print('Veriye gore: Cascade pattern bazi sistemlerde ciddiyet artirirken, bazilarinda artirmiyor.')
print('Bakim turu (degisim vs tamir) bilgimiz olmadigi icin sebep yorumlanmaz.')
print('Sadece raporlanir: hangi sistemler cascade-sensitif, hangileri degil.')


=== KATEGORI BAZLI: CASCADE vs STAND-ALONE CIDDIYET ===
Kategori                             cascade_n  ort_skor_C  stand_n   ort_skor_S  fark    p
✓ SOĞUTMA SİSTEMİ ARIZASI              1048     3.456       5334    3.317     +0.138  0.0243
✓ KAPI ARIZALARI                       1042     3.180       5103    3.002     +0.178  0.0001
✓ ELEKTRİK SİSTEMİ ARIZALARI            900     3.379       5035    2.964     +0.414  0.0000
✓ MOTOR ARIZALARI                       871     4.549       4766    4.304     +0.245  0.0001
  KAROSER ARIZALARI                     456     2.845       3743    2.818     +0.027  0.8624
✓ KLİMA SİSTEMİ ARIZALARI               674     3.055       3269    2.841     +0.215  0.0001
  FREN ŞİKAYETLERİ                      504     5.076       3343    4.958     +0.117  0.3600
  SÜSPANSİYON SİSTEMİ ARIZALARI         512     4.067       2875    3.959     +0.108  0.1583
✓ OTOMATİK ŞANZIMAN ARIZALARI           554     4.659       2670    4.427     +0.232  0.0037
✓ ISITMA SİSTEM

---

## 17. Fırtına Paradoksu Açıklaması
Bölüm 6'da fırtına yaşayan araçlar daha ciddi değil çıktı. Hangi sistemler fırtına yaratıyor?

In [17]:
# BOLUM 17: Firtina Paradoksu Aciklamasi
# Cell 11'de: Firtinali araclar daha ciddi DEGIL (p=0.98). Garip.
# Soru: Firtina hangi sistemlerin tekrarindan olusuyor?
# DIKKAT: Sebep yorumlanmaz (bakim turu bilinmiyor), sadece raporlanir.

# 24s icinde 3+ ariza yapan araclar
firtina_arac_set = set()
for kapino, sub in df.groupby('KAPINO'):
    tarihler = sub['OLAYTARIHI'].sort_values().values
    for i in range(len(tarihler)):
        pencere = tarihler[(tarihler >= tarihler[i]) &
                            (tarihler <= tarihler[i] + np.timedelta64(24, 'h'))]
        if len(pencere) >= 3:
            firtina_arac_set.add(kapino)
            break

# Bu araclarin TUM arizalarini al
firtina_df = df[df['KAPINO'].isin(firtina_arac_set)]
diger_df = df[~df['KAPINO'].isin(firtina_arac_set)]

print(f'Firtinali arac sayisi: {len(firtina_arac_set):,}')
print(f'Firtinali ariza sayisi: {len(firtina_df):,}')

# Firtinali araclarda kategori dagilimi
print()
print('=== FIRTINALI ARACLARDA EN COK GORULEN SISTEMLER ===')
firt_dist = firtina_df['ARIZAUSTKODTANIM'].value_counts(normalize=True).head(10) * 100
gen_dist = df['ARIZAUSTKODTANIM'].value_counts(normalize=True) * 100

print(f'{"Kategori":35s}  firt_%   genel_%   Lift')
for kat in firt_dist.index:
    f = firt_dist[kat]
    g = gen_dist.get(kat, 0)
    lift = f / g if g > 0 else 0
    print(f'{str(kat)[:35]:35s}  {f:6.2f}  {g:6.2f}    {lift:.2f}')

# Bu firtinali araclarda CIDDI ARIZA orani ne?
print()
print('=== FIRTINALI ARACLARIN ARIZA CIDDIYETI DETAYI ===')
print(f'Firtinali ariza ort_skor:  {firtina_df["ciddiyet_skoru"].mean():.3f}')
print(f'Diger ariza ort_skor:      {diger_df["ciddiyet_skoru"].mean():.3f}')
print(f'Firtinali ciddi_ariza %:    {firtina_df["ciddi_ariza"].mean()*100:.1f}')
print(f'Diger ciddi_ariza %:        {diger_df["ciddi_ariza"].mean()*100:.1f}')

# Firtinali sistemler hangileri?
# Eger firtina yuksek lift'le KAMERA/ISITMA gibi non-ciddi sistemlere kaymissa, paradoks acıklanır
firt_kategoriler = firt_dist.index[:5].tolist()
print()
print('=== TOP FIRTINA SISTEMLERINDE CIDDI_ARIZA ORANI ===')
print(f'{"Sistem":35s}  toplam_ciddi%')
for kat in firt_kategoriler:
    sub = df[df['ARIZAUSTKODTANIM'] == kat]
    if len(sub) >= 100:
        ciddi_p = sub['ciddi_ariza'].mean() * 100
        print(f'{str(kat)[:35]:35s}  {ciddi_p:.1f}%')

print()
print('=== YORUM (veriden) ===')
print('Firtina patternine bakarak hangi sistemler tekrar tekrar arizalanıyor goruluyor.')
print('Bakim turu (degisim/tamir) bilinmeden sebep cikarilamaz.')
print('Veri raporu: firtina yaratan sistemlerin ortalama ciddiyet skoru genel filodan')
print('farkli olabilir veya olmayabilir — yukarida sayisal sonuc var.')


Firtinali arac sayisi: 613
Firtinali ariza sayisi: 15,754

=== FIRTINALI ARACLARDA EN COK GORULEN SISTEMLER ===
Kategori                             firt_%   genel_%   Lift
KAPI ARIZALARI                        12.71   10.49    1.21
SOĞUTMA SİSTEMİ ARIZASI               11.11   10.90    1.02
ELEKTRİK SİSTEMİ ARIZALARI            10.64   10.14    1.05
MOTOR ARIZALARI                        9.83    9.63    1.02
KLİMA SİSTEMİ ARIZALARI                6.81    6.73    1.01
KAROSER ARIZALARI                      6.68    7.17    0.93
SÜSPANSİYON SİSTEMİ ARIZALARI          6.18    5.78    1.07
FREN ŞİKAYETLERİ                       6.09    6.57    0.93
OTOMATİK ŞANZIMAN ARIZALARI            5.71    5.51    1.04
ISITMA SİSTEMİ                         4.33    4.55    0.95

=== FIRTINALI ARACLARIN ARIZA CIDDIYETI DETAYI ===
Firtinali ariza ort_skor:  3.618
Diger ariza ort_skor:      3.607
Firtinali ciddi_ariza %:    39.2
Diger ciddi_ariza %:        37.6

=== TOP FIRTINA SISTEMLERINDE CIDDI_ARIZA 

---

## 18. Ordinal/Bant Feature'lar (Tree Model İçin)
Lineer korelasyon zayıf, ANOVA güçlü → tree model için ordinal feature'lar uygulanır.

In [18]:
# BOLUM 18: Ordinal/Bant Feature'lar (Tree Model Icin)
# Bolum 11'de Pearson r zayifti (max 0.06).
# Ama Bolum 12 ANOVA F=65 cok guclu (gecmis_ciddi_n bantlari arasinda fark var).
# Bu non-linear/monotonic iliski demek — tree-based modeller (XGBoost) bunu yakalar.
# Yeni ordinal feature'lar uretelim.

# Ordinal bantlar: gecmis_ciddi_n
df_feat['ord_gecmis_ciddi'] = pd.cut(df_feat['gecmis_ciddi_n'],
    bins=[-0.1, 0.5, 2.5, 5.5, 1000],
    labels=[0, 1, 2, 3]).astype(int)

# Ordinal: ariza_son_7g
df_feat['ord_son7g'] = pd.cut(df_feat['ariza_son_7g'],
    bins=[-0.1, 0.5, 1.5, 2.5, 100],
    labels=[0, 1, 2, 3]).astype(int)

# Ordinal: son_ariza_gun (kucuk = yakın zaman = risk)
df_feat['ord_son_ariza'] = pd.cut(df_feat['son_ariza_gun'],
    bins=[-0.1, 1, 7, 30, 1000],
    labels=[3, 2, 1, 0]).astype(int)  # 3=cok yakin, 0=hic ariza yok

# Ordinal: farkli_kat_30g
df_feat['ord_farkli_kat'] = pd.cut(df_feat['farkli_kat_30g'],
    bins=[-0.1, 0.5, 2.5, 5.5, 100],
    labels=[0, 1, 2, 3]).astype(int)

# Composite "cascade risk skoru" (toplama)
df_feat['cascade_risk_skor'] = (
    df_feat['ord_gecmis_ciddi'] +
    df_feat['ord_son7g'] +
    df_feat['ord_son_ariza'] +
    df_feat['ord_farkli_kat']
)

print('=== ORDINAL FEATURE\'LAR ===')
print(df_feat[['ord_gecmis_ciddi','ord_son7g','ord_son_ariza','ord_farkli_kat','cascade_risk_skor']].describe().round(2).to_string())

# Bunlarin korelasyonu (Spearman daha uygun)
print()
print('=== ORDINAL FEATURE x CIDDIYET KORELASYON ===')
yeni_features = ['ord_gecmis_ciddi','ord_son7g','ord_son_ariza','ord_farkli_kat','cascade_risk_skor']
for f in yeni_features:
    r, p = stats.pearsonr(df_feat[f], df_feat['ciddiyet_skoru'])
    rs, ps = stats.spearmanr(df_feat[f], df_feat['ciddiyet_skoru'])
    r_cb, p_cb = stats.pointbiserialr(df_feat['ciddi_ariza'], df_feat[f])
    print(f'{f:22s}  Pearson r={r:+.4f}  Spearman rs={rs:+.4f}  CiddiAriza r={r_cb:+.4f}')

# Composite skor bant analizi (tree model'in nasil kullanabilecegini gosterir)
print()
print('=== CASCADE_RISK_SKOR BANTLARINA GORE CIDDIYET ===')
df_feat['crs_bant'] = pd.cut(df_feat['cascade_risk_skor'],
    bins=[-0.1, 2, 5, 8, 100],
    labels=['Cok Dusuk (0-2)', 'Dusuk (3-5)', 'Yuksek (6-8)', 'Cok Yuksek (9+)'])
print(df_feat.groupby('crs_bant', observed=False).agg(
    n=('cascade_risk_skor','count'),
    ort_skor=('ciddiyet_skoru','mean'),
    ciddi_p=('ciddi_ariza','mean'),
).round(3).to_string())

# ANOVA
gruplar = [g['ciddiyet_skoru'].values for _, g in df_feat.groupby('crs_bant', observed=False) if len(g) >= 50]
if len(gruplar) >= 2:
    f, p = stats.f_oneway(*gruplar)
    print(f'\nANOVA F={f:.2f}, p={p:.6f}')
    if p < 0.05:
        print('VERIYE GORE: cascade_risk_skor bandlari arasinda ANLAMLI ciddiyet farki var.')
        print('  -> Tree model (XGBoost) bu non-linear yapiyi yakalamaya uygun.')


=== ORDINAL FEATURE'LAR ===
       ord_gecmis_ciddi  ord_son7g  ord_son_ariza  ord_farkli_kat  cascade_risk_skor
count          58559.00   58559.00       58559.00        58559.00           58559.00
mean               1.78       0.89           1.59            1.40               5.66
std                1.01       0.98           0.88            0.76               2.68
min                0.00       0.00           0.00            0.00               0.00
25%                1.00       0.00           1.00            1.00               4.00
50%                2.00       1.00           2.00            1.00               6.00
75%                3.00       1.00           2.00            2.00               8.00
max                3.00       3.00           3.00            3.00              12.00

=== ORDINAL FEATURE x CIDDIYET KORELASYON ===
ord_gecmis_ciddi        Pearson r=+0.0554  Spearman rs=+0.0724  CiddiAriza r=+0.0724
ord_son7g               Pearson r=+0.0379  Spearman rs=+0.0526  CiddiAriza 

---

## 19. Suppressor Effect Açıklaması (Cell 17 Paradoks)
Cell 17'deki M3 cascade negatif anlamlılığı suppressor effect — açıklama ve karar.

In [19]:
# BOLUM 19: Suppressor Effect Aciklamasi (Cell 17 Paradoks)
# Cell 17'de M3 modelinde cascade_orani NEGATIF anlamli cikti (k=-0.36, p=0.003).
# Bu paradoks degil "suppressor effect" — kontrol degiskenler dahil edilince yon degisebilir.

# Ham korelasyon vs partial
print('=== SUPPRESSOR EFFECT KONTROLU ===')

# Ham korelasyon
r_ham, p_ham = stats.pearsonr(arac_cas_full['cascade_orani'], arac_cas_full['ort_skor'])
print(f'Ham korelasyon cascade x ort_skor: r={r_ham:+.3f}, p={p_ham:.4f}')

# Yas ile ham korelasyon (kontrol degiskeni)
r_y_c, p_y_c = stats.pearsonr(arac_cas_full['arac_yasi'], arac_cas_full['cascade_orani'])
r_y_s, p_y_s = stats.pearsonr(arac_cas_full['arac_yasi'], arac_cas_full['ort_skor'])
print(f'Yas x Cascade:  r={r_y_c:+.3f}, p={p_y_c:.4f}')
print(f'Yas x Skor:     r={r_y_s:+.3f}, p={p_y_s:.4f}')

# Sefer ile
r_s_c, p_s_c = stats.pearsonr(arac_cas_full['toplam_sefer'].fillna(0), arac_cas_full['cascade_orani'])
r_s_s, p_s_s = stats.pearsonr(arac_cas_full['toplam_sefer'].fillna(0), arac_cas_full['ort_skor'])
print(f'Sefer x Cascade: r={r_s_c:+.3f}, p={p_s_c:.4f}')
print(f'Sefer x Skor:    r={r_s_s:+.3f}, p={p_s_s:.4f}')

# Garaj VAR mi yok mu test edelim
print()
print('=== GARAJ KONTROL ETKISI ===')
# Sadece arac_cas_full ile yapilmali
import statsmodels.api as sm
from statsmodels.formula.api import ols

regdf2 = arac_cas_full.dropna(subset=['toplam_sefer','arac_yasi','GARAJ']).copy()
regdf2['log_sefer'] = np.log1p(regdf2['toplam_sefer'])

# 1. SIRADAN cascade -> skor
m_a = ols('ort_skor ~ cascade_orani', data=regdf2).fit()
# 2. + yas
m_b = ols('ort_skor ~ cascade_orani + arac_yasi', data=regdf2).fit()
# 3. + sefer
m_c = ols('ort_skor ~ cascade_orani + arac_yasi + log_sefer', data=regdf2).fit()
# 4. + garaj (full)
m_d = ols('ort_skor ~ cascade_orani + arac_yasi + log_sefer + C(GARAJ)', data=regdf2).fit()

print(f'{"Model":40s}  cascade_k  cascade_p   R^2')
for ad, m in [('A: sadece cascade', m_a), ('B: +yas', m_b), ('C: +yas+sefer', m_c), ('D: +yas+sefer+garaj', m_d)]:
    k = m.params['cascade_orani']
    p = m.pvalues['cascade_orani']
    print(f'{ad:40s}  {k:+.4f}    {p:.4f}    {m.rsquared:.3f}')

print()
print('=== YORUM (suppressor effect) ===')
print('Cascade katsayi modeller ilerledikce kontrolleri ile yon degistiriyor:')
print(f'  A (kontrolsuz): {m_a.params["cascade_orani"]:+.3f}')
print(f'  D (full kontrol): {m_d.params["cascade_orani"]:+.3f}')
print()
print('Bu suppressor effect oldugunu gosterir:')
print(' - Cascade orani garaj ve sefer ile pozitif korelasyon icinde.')
print(' - Garaj/sefer kontrol edildiginde, cascade ile skor arasindaki gercek')
print('   net iliski ortaya cikiyor (potansiyel olarak negatif veya farkli yonlu).')
print(' - "Cascade az -> ariza ciddi degil" diye yorumlanmamali; bu sadece')
print('   garaj ve sefer dagiliminin etkisi.')
print()
print('SONUC: Cascade_orani tek basina ML feature olarak GUVENSIZ.')
print('  Bunun yerine ordinal bantsal feature\'lar (cascade_risk_skor) kullanilmali.')


=== SUPPRESSOR EFFECT KONTROLU ===
Ham korelasyon cascade x ort_skor: r=-0.005, p=0.7625
Yas x Cascade:  r=+0.030, p=0.0843
Yas x Skor:     r=+0.138, p=0.0000
Sefer x Cascade: r=-0.141, p=0.0000
Sefer x Skor:    r=-0.122, p=0.0000

=== GARAJ KONTROL ETKISI ===
Model                                     cascade_k  cascade_p   R^2
A: sadece cascade                         -0.0375    0.7625    0.000
B: +yas                                   -0.0670    0.5858    0.019
C: +yas+sefer                             -0.1859    0.1314    0.033
D: +yas+sefer+garaj                       -0.3640    0.0025    0.210

=== YORUM (suppressor effect) ===
Cascade katsayi modeller ilerledikce kontrolleri ile yon degistiriyor:
  A (kontrolsuz): -0.037
  D (full kontrol): -0.364

Bu suppressor effect oldugunu gosterir:
 - Cascade orani garaj ve sefer ile pozitif korelasyon icinde.
 - Garaj/sefer kontrol edildiginde, cascade ile skor arasindaki gercek
   net iliski ortaya cikiyor (potansiyel olarak negatif veya 

---

## 20. Sistem Bazlı Cascade Lift Feature (Final Test)
Cell 31'de tespit edilen "sistem cascade ciddiyet artış" verilerini ML feature olarak türet ve test et.

In [20]:
# BOLUM 20: Sistem Bazli Cascade Lift Feature
# Cell 31'de hangi sistemlerde cascade ciddiyet artirdigi tespit edildi.
# Bu bilgiyi feature olarak turetip ML icin test edelim.

# Cell 31'in sonuclarini yeniden hesapla (sistem -> cascade lift)
# Lift = ort_skor_cascade / ort_skor_standalone
sistem_lift = {}
for kat in df_feat['ARIZAUSTKODTANIM'].value_counts().head(25).index:
    sub = df_feat[df_feat['ARIZAUSTKODTANIM'] == kat]
    c_sub = sub[sub['cascade_24s'] == 1]
    s_sub = sub[sub['cascade_24s'] == 0]
    if len(c_sub) < 20 or len(s_sub) < 50:
        sistem_lift[kat] = 1.0  # default
        continue
    skor_c = c_sub['ciddiyet_skoru'].mean()
    skor_s = s_sub['ciddiyet_skoru'].mean()
    lift = skor_c / skor_s if skor_s > 0 else 1.0
    sistem_lift[kat] = lift

print('=== SISTEM CASCADE LIFT (ciddiyet artis orani) ===')
for kat, lift in sorted(sistem_lift.items(), key=lambda x: -x[1])[:15]:
    print(f'  {str(kat)[:40]:40s}: lift={lift:.3f}')

# Feature olarak ekle
# - sistem_cascade_lift: o kategorinin cascade ciddiyet artis orani
# - sistem_cas_lift_x_cascade24s: lift x cascade flag interaksiyonu

df_feat['sistem_cas_lift'] = df_feat['ARIZAUSTKODTANIM'].map(sistem_lift).fillna(1.0)
df_feat['lift_x_cascade'] = df_feat['sistem_cas_lift'] * df_feat['cascade_24s']

# Korelasyon testleri
print()
print('=== YENI FEATURE KORELASYON ===')
yeni_features_2 = ['sistem_cas_lift', 'lift_x_cascade']
for f in yeni_features_2:
    r, p = stats.pearsonr(df_feat[f], df_feat['ciddiyet_skoru'])
    rs, ps = stats.spearmanr(df_feat[f], df_feat['ciddiyet_skoru'])
    rc, pc = stats.pointbiserialr(df_feat['ciddi_ariza'], df_feat[f])
    print(f'{f:25s}  Pearson r={r:+.4f}  Spearman rs={rs:+.4f}  CiddiAriza r={rc:+.4f}  (p_pearson={p:.4f})')

# Bant analizi sistem_cas_lift
df_feat['scl_bant'] = pd.cut(df_feat['sistem_cas_lift'],
    bins=[0, 1.0, 1.05, 1.10, 2.0],
    labels=['Yok (≤1.0)', 'Hafif (1.0-1.05)', 'Orta (1.05-1.10)', 'Yuksek (1.10+)'])
print()
print('=== SISTEM_CAS_LIFT BANTLARINA GORE ===')
print(df_feat.groupby('scl_bant', observed=False).agg(
    n=('sistem_cas_lift', 'count'),
    ort_skor=('ciddiyet_skoru', 'mean'),
    ciddi_p=('ciddi_ariza', 'mean'),
).round(3).to_string())

gruplar = [g['ciddiyet_skoru'].values for _, g in df_feat.groupby('scl_bant', observed=False) if len(g) >= 50]
if len(gruplar) >= 2:
    f, p = stats.f_oneway(*gruplar)
    print(f'\nANOVA F={f:.2f}, p={p:.6f}')

# Lift_x_cascade — sadece cascade=1 olanlar icin etkili
print()
print('=== LIFT_X_CASCADE = SISTEM_LIFT * CASCADE_24S ===')
print('Bu feature sadece cascade arızalarinda ciddiyet artis sinyali verir')
sub_cas = df_feat[df_feat['cascade_24s'] == 1]
print(f'Cascade kayit: {len(sub_cas):,}')
r, p = stats.pearsonr(sub_cas['sistem_cas_lift'], sub_cas['ciddiyet_skoru'])
print(f'Cascade icinde sistem_lift x ciddiyet: r={r:+.4f}, p={p:.4f}')

print()
print('=== KARAR ===')
print('sistem_cas_lift, lift_x_cascade feature\'lari ML icin test edildi.')
print('Korelasyon degerlerine bakarak ekleme karari verilir.')

# En guclu feature listesi (Bolum 21'i guncelle)
print()
print('=== TOPLU ML FEATURE SIRALAMASI (en guclu) ===')
tum_features = {
    'cascade_risk_skor':  df_feat['cascade_risk_skor'],
    'ord_gecmis_ciddi':   df_feat['ord_gecmis_ciddi'],
    'ord_son7g':          df_feat['ord_son7g'],
    'ord_son_ariza':      df_feat['ord_son_ariza'],
    'ord_farkli_kat':     df_feat['ord_farkli_kat'],
    'sistem_cas_lift':    df_feat['sistem_cas_lift'],
    'lift_x_cascade':     df_feat['lift_x_cascade'],
    'gecmis_ciddi_n':     df_feat['gecmis_ciddi_n'],
    'ciddi_son_30g':      df_feat['ciddi_son_30g'],
    'ciddi_son_7g':       df_feat['ciddi_son_7g'],
}

sira_list = []
for fad, fv in tum_features.items():
    rs, ps = stats.spearmanr(fv, df_feat['ciddiyet_skoru'])
    rc, pc = stats.pointbiserialr(df_feat['ciddi_ariza'], fv)
    sira_list.append({'feature': fad, 'rs_skor': rs, 'rc_ciddi': rc})

sira_df = pd.DataFrame(sira_list).sort_values('rs_skor', key=abs, ascending=False)
print(f'\n{"Feature":22s}  Spearman_rs   CiddiAriza_r')
for _, row in sira_df.iterrows():
    print(f'{row.feature:22s}  {row.rs_skor:+.4f}      {row.rc_ciddi:+.4f}')


=== SISTEM CASCADE LIFT (ciddiyet artis orani) ===
  İLAVE DİREKSİYON SİSTEMİ ARIZALARI      : lift=1.247
  YANGIN İKAZ ARIZALARI                   : lift=1.148
  ELEKTRİK SİSTEMİ ARIZALARI              : lift=1.140
  BASINÇLI MOTOR YAĞI SİSTEMİ             : lift=1.088
  YAKIT ve ENJEKSİYON ARIZALARI           : lift=1.087
  ISITMA SİSTEMİ                          : lift=1.080
  KAMERA SİSTEMİ ARIZALARI                : lift=1.077
  KLİMA SİSTEMİ ARIZALARI                 : lift=1.076
  DİREKSİYON ARIZALARI                    : lift=1.066
  BASINÇLI HAVA DONANIMI ARIZALARI        : lift=1.061
  KAPI ARIZALARI                          : lift=1.059
  MOTOR ARIZALARI                         : lift=1.057
  OTOMATİK ŞANZIMAN ARIZALARI             : lift=1.052
  SOĞUTMA SİSTEMİ ARIZASI                 : lift=1.042
  BASINÇLI HAVA HATTI ARIZASI             : lift=1.029

=== YENI FEATURE KORELASYON ===
sistem_cas_lift            Pearson r=-0.1804  Spearman rs=-0.1841  CiddiAriza r=-0.1291  (p

---

## 21. Common Cause vs True Cascade Ayrımı
Mevcut cascade tanımımız sequential pattern içeriyor. Bu pattern iki sebepten gelebilir:
- **True Cascade:** A bozulması B'yi tetikledi (mekanik)
- **Common Cause:** Dışarıdan ortak sebep (yol, hava, sürüş) hem A hem B'yi etkiledi

Test: Aynı zaman + aynı HAT + farklı araç + aynı sistem = common cause sinyali.

In [21]:
# BOLUM 21: Common Cause vs True Cascade Ayrimi
# Mevcut "cascade" tanimimiz sequential (ayni arac A->B 24s). Ama bu pattern iki sebepten gelebilir:
#  - True Cascade: A bozulmasi B'yi tetikledi
#  - Common Cause: Disaridan ortak sebep (yol, hava, surus) hem A hem B'yi etkiledi
# Test: Ayni zaman + ayni HAT + farkli arac + ayni sistem arızasi = common cause sinyali

from datetime import timedelta

# 1. AYNI SAAT + AYNI HAT + FARKLI ARAC + AYNI SISTEM CLUSTER
# Saatlik bin'le grupla
df_cc = df.copy()
df_cc['saat_bin'] = df_cc['OLAYTARIHI'].dt.floor('h')  # saatlik kova

# (saat_bin, HATKODU, ARIZAUSTKODTANIM) gruplari
grup_say = df_cc.groupby(['saat_bin', 'HATKODU', 'ARIZAUSTKODTANIM']).agg(
    n_ariza=('KAPINO', 'count'),
    n_farkli_arac=('KAPINO', 'nunique'),
).reset_index()

# Common cause sinyali: 1 saat + 1 hat + ayni sistem + 2+ farkli arac
common_cluster = grup_say[grup_say['n_farkli_arac'] >= 2]

print('=== COMMON CAUSE TESPITI ===')
print(f'Toplam (saat, hat, sistem) grup: {len(grup_say):,}')
print(f'Common cause cluster (>=2 farkli arac): {len(common_cluster):,}')
print(f'Bu cluster\'lardaki toplam ariza: {common_cluster["n_ariza"].sum():,}')
print()

# Top cluster orneklemi
print('=== EN BUYUK COMMON CAUSE CLUSTERLARI ===')
print(common_cluster.nlargest(10, 'n_farkli_arac').to_string(index=False))

# Bu pattern cascade tanimimiza ne kadar dahil?
# Cascade arizalari (Cell 4 sonrasi): df_cc icinde cascade==True olanlar
df_cc['onceki_t'] = df_cc.groupby('KAPINO')['OLAYTARIHI'].shift(1)
df_cc['saat_fark'] = (df_cc['OLAYTARIHI'] - df_cc['onceki_t']).dt.total_seconds() / 3600
df_cc['is_cascade'] = df_cc['saat_fark'] < 24
cascade_kayitlar = df_cc[df_cc['is_cascade']]

# Cascade icinde common cause olan kayit
# Yontem: cascade kaydinin oldugu saat+hat+sistem grubunda baska arac var mi?
print()
print('=== CASCADE ARIZALARININ COMMON CAUSE ILE OVERLAP ORANI ===')
cascade_grup = cascade_kayitlar.merge(
    grup_say[grup_say['n_farkli_arac'] >= 2][['saat_bin','HATKODU','ARIZAUSTKODTANIM']].assign(common=True),
    on=['saat_bin','HATKODU','ARIZAUSTKODTANIM'], how='left'
)
cascade_grup['common'] = cascade_grup['common'].fillna(False)
common_oran = cascade_grup['common'].mean()
n_common_cascade = cascade_grup['common'].sum()
print(f'Cascade ariza toplam: {len(cascade_kayitlar):,}')
print(f'Common cause overlap (cascade + ayni hat-saat-sistem cluster): {n_common_cascade:,} ({common_oran*100:.1f}%)')

# Yorum esikleri
print()
print('=== YORUM (veriye gore) ===')
if common_oran < 0.10:
    print(f'%{common_oran*100:.1f} common cause overlap < %10')
    print('-> Cascade\'lerimiz BUYUK OLCUDE TRUE SEQUENTIAL (gercek tetikleme)')
    print('   ML feature\'lar dogru yorumlanir.')
elif common_oran < 0.30:
    print(f'%{common_oran*100:.1f} common cause overlap %10-30 arasi')
    print('-> Karisik durum. Cascade\'in bir kismi common cause kaynakli.')
    print('   ML feature\'lari dikkatli yorumla — bazi pattern\'lar dis sebepli olabilir.')
else:
    print(f'%{common_oran*100:.1f} common cause overlap > %30')
    print('-> Common cause DOMINANT. Cascade analizi yorumu sinirli.')
    print('   ML feature\'lar gercek tetikleme degil, ortak cevre faktoru yansitiyor olabilir.')

# Detay: hangi sistemler en cok common cluster yapiyor?
print()
print('=== EN COK COMMON CAUSE PATTERN GOSTEREN SISTEMLER ===')
common_kategori = common_cluster.groupby('ARIZAUSTKODTANIM').agg(
    cluster_n=('saat_bin', 'count'),
    toplam_ariza=('n_ariza', 'sum'),
    ort_farkli_arac=('n_farkli_arac', 'mean'),
).sort_values('cluster_n', ascending=False).head(10)
print(common_kategori.round(2).to_string())


=== COMMON CAUSE TESPITI ===
Toplam (saat, hat, sistem) grup: 56,930
Common cause cluster (>=2 farkli arac): 1,407
Bu cluster'lardaki toplam ariza: 2,967

=== EN BUYUK COMMON CAUSE CLUSTERLARI ===
           saat_bin HATKODU            ARIZAUSTKODTANIM  n_ariza  n_farkli_arac
2025-04-21 17:00:00    34AS     KLİMA SİSTEMİ ARIZALARI        5              5
2025-02-27 17:00:00     34G              KAPI ARIZALARI        4              4
2025-03-14 14:00:00    34BZ     KLİMA SİSTEMİ ARIZALARI        4              4
2025-03-15 06:00:00     34G     SOĞUTMA SİSTEMİ ARIZASI        4              4
2025-05-23 17:00:00    34AS     KLİMA SİSTEMİ ARIZALARI        4              4
2025-05-24 14:00:00    34BZ     KLİMA SİSTEMİ ARIZALARI        4              4
2025-06-20 12:00:00    34BZ     KLİMA SİSTEMİ ARIZALARI        4              4
2025-06-24 16:00:00    34BZ     KLİMA SİSTEMİ ARIZALARI        4              4
2025-06-24 16:00:00    34BZ OTOMATİK ŞANZIMAN ARIZALARI        4              4
202

---

## 22. Coğrafi-Zamansal Cluster Testi
Common cause için coğrafi test: ENLEM/BOYLAM koordinatlarıyla ~1 km grid + 1 saat penceresi.
Birden fazla araç aynı bölgede aynı saat içinde aynı sistemde arızalandıysa → dış etki olabilir.

In [22]:
# BOLUM 22: Cografi-Zamansal Cluster Testi
# Common cause icin alternatif test: kucuk cografi alan + kucuk zaman penceresi icinde
# birden cok arac AYNI sistem arızasi yapti mi?
# ENLEM/BOYLAM kullanarak coğrafi cluster tespit edelim.

from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# ENLEM/BOYLAM ariza_model.csv'de yok, ariza_temiz.csv'de var
df_temiz = pd.read_csv('../panel_data/temiz_veri/ariza_temiz.csv', low_memory=False)
df_temiz['OLAYTARIHI'] = pd.to_datetime(df_temiz['OLAYTARIHI'], format='mixed')

# ENLEM/BOYLAM string olabilir, numeric'e cevir (virgul -> nokta da)
df_temiz['ENLEM']  = pd.to_numeric(df_temiz['ENLEM'].astype(str).str.replace(',', '.'),  errors='coerce')
df_temiz['BOYLAM'] = pd.to_numeric(df_temiz['BOYLAM'].astype(str).str.replace(',', '.'), errors='coerce')

df_geo = df_temiz[(df_temiz['ENLEM'].notna()) & (df_temiz['BOYLAM'].notna())].copy()
df_geo = df_geo[(df_geo['ENLEM'].between(40, 42)) & (df_geo['BOYLAM'].between(28, 30))]
print(f'Cografi koordinati olan ariza: {len(df_geo):,} / {len(df_temiz):,}')

# Saatlik bin + grid bin
# 0.01 derece grid = ~1 km
df_geo['saat_bin'] = df_geo['OLAYTARIHI'].dt.floor('h')
df_geo['lat_grid'] = (df_geo['ENLEM']  * 100).round() / 100   # ~1 km
df_geo['lon_grid'] = (df_geo['BOYLAM'] * 100).round() / 100

# (saat, lat_grid, lon_grid, sistem) → farkli arac sayisi
geo_cluster = df_geo.groupby(['saat_bin', 'lat_grid', 'lon_grid', 'ARIZAUSTKODTANIM']).agg(
    n_ariza=('KAPINO', 'count'),
    n_farkli_arac=('KAPINO', 'nunique'),
).reset_index()

cografi_common = geo_cluster[geo_cluster['n_farkli_arac'] >= 2]

print()
print('=== COGRAFI-ZAMANSAL CLUSTER (1 saat + ~1 km + ayni sistem) ===')
print(f'Toplam grup: {len(geo_cluster):,}')
print(f'Common cluster (>=2 farkli arac): {len(cografi_common):,}')
print(f'Bu cluster\'lardaki toplam ariza: {cografi_common["n_ariza"].sum():,}')

# Cascade arizalari icinde overlap
df_cc_geo = df_geo.copy()
df_cc_geo['onceki_t'] = df_cc_geo.groupby('KAPINO')['OLAYTARIHI'].shift(1)
df_cc_geo['saat_fark'] = (df_cc_geo['OLAYTARIHI'] - df_cc_geo['onceki_t']).dt.total_seconds() / 3600
df_cc_geo['is_cascade'] = df_cc_geo['saat_fark'] < 24
cascade_geo = df_cc_geo[df_cc_geo['is_cascade']]

cascade_geo_merged = cascade_geo.merge(
    cografi_common[['saat_bin','lat_grid','lon_grid','ARIZAUSTKODTANIM']].assign(geo_common=True),
    on=['saat_bin','lat_grid','lon_grid','ARIZAUSTKODTANIM'], how='left'
)
cascade_geo_merged['geo_common'] = cascade_geo_merged['geo_common'].fillna(False)
geo_overlap = cascade_geo_merged['geo_common'].mean()
print()
print(f'Cascade icinde cografi-zamansal common overlap: {cascade_geo_merged["geo_common"].sum():,} ({geo_overlap*100:.1f}%)')

print()
print('=== EN BUYUK COGRAFI-ZAMANSAL CLUSTERLARI ===')
print(cografi_common.nlargest(10, 'n_farkli_arac')[
    ['saat_bin','lat_grid','lon_grid','ARIZAUSTKODTANIM','n_ariza','n_farkli_arac']
].to_string(index=False))

# Yorum
print()
print('=== YORUM ===')
if geo_overlap < 0.10:
    print(f'%{geo_overlap*100:.1f} cografi-zamansal common overlap < %10')
    print('-> Cascade\'ler cografi olarak izole (true sequential dominant)')
elif geo_overlap < 0.30:
    print(f'%{geo_overlap*100:.1f} cografi-zamansal overlap %10-30')
    print('-> Bir kismi yol/cevre kaynakli olabilir, kismi true cascade')
else:
    print(f'%{geo_overlap*100:.1f} > %30 — yuksek common cause sinyali')

# Birinci test ile karsilastir
print()
print('=== IKI TEST OZETI ===')
print(f'Hat bazli common cause (Bolum 21):       %{common_oran*100:.1f}')
print(f'Cografi-zamansal common cause (Bolum 22): %{geo_overlap*100:.1f}')

# Final yargi
max_common = max(common_oran, geo_overlap)
print()
print('=== FINAL YARGI ===')
if max_common < 0.10:
    print('Cascade\'lerimiz TRUE SEQUENTIAL — common cause minimum.')
    print('Cell 1-20 analizleri ve ML feature\'lar dogru yorumlanabilir.')
elif max_common < 0.30:
    print('Cascade\'lerin bir kismi common cause olabilir, ama coğunluk true sequential.')
    print('ML feature\'lar genelde dogru, ama kategoriye gore varyans olabilir.')
else:
    print('Common cause dominant olabilir.')
    print('ML feature yorumlarini sinirla, cascade analizi gercek sequential olmayabilir.')


Cografi koordinati olan ariza: 58,990 / 59,050

=== COGRAFI-ZAMANSAL CLUSTER (1 saat + ~1 km + ayni sistem) ===
Toplam grup: 58,282
Common cluster (>=2 farkli arac): 647
Bu cluster'lardaki toplam ariza: 1,344

Cascade icinde cografi-zamansal common overlap: 241 (2.7%)

=== EN BUYUK COGRAFI-ZAMANSAL CLUSTERLARI ===
           saat_bin  lat_grid  lon_grid           ARIZAUSTKODTANIM  n_ariza  n_farkli_arac
2025-06-26 12:00:00     41.02     28.62    KLİMA SİSTEMİ ARIZALARI        5              5
2025-01-15 10:00:00     41.02     28.63             KAPI ARIZALARI        4              4
2025-03-26 06:00:00     40.99     29.04 ELEKTRİK SİSTEMİ ARIZALARI        4              4
2025-05-23 13:00:00     40.99     29.04    SOĞUTMA SİSTEMİ ARIZASI        4              4
2025-06-19 16:00:00     40.99     29.04    KLİMA SİSTEMİ ARIZALARI        4              4
2025-01-15 19:00:00     40.99     29.04             KAPI ARIZALARI        3              3
2025-01-24 06:00:00     40.99     29.04 ELEKTRİ